Steigung (Segment)

In [1]:
import osmnx as ox
import math
import pandas as pd
import numpy as np
from osgeo import gdal
# OSMnx konfigurieren
ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing:markings',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

latitude, longitude =47.976121765929626,7.7773975938558655

 #48.80520352412432,9.169831239419484

meters_per_lat = 111320
meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
delta_lat = 5000/ meters_per_lat
delta_lon = 5000/ meters_per_lon
north = latitude + delta_lat
south = latitude
east = longitude + delta_lon
west = longitude
bbox = (west, south, east, north)

G1 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

# Entfernen von "service" Edges

service_edges = [(u, v, k) for u, v, k, d in G1.edges(keys=True, data=True) if d.get('highway') == 'service']

G1.remove_edges_from(service_edges)
G_original = copy.deepcopy(G1)
#Steigung OSMnx Funktion definieren, um Steigungprozente an die Segmente zu ermitteln

def add_elevation_and_slope(G1, raster_path):
    G1 = ox.elevation.add_node_elevations_raster(G1, raster_path)
    G1 = ox.elevation.add_edge_grades(G1, add_absolute=True)
    return G1
#Rasterdatei mit Höhenlinien werden eingespielt
raster_path = "srtm_germany_dtm.tif"
#Funktion wird auf diese Rasterdatei angewandt
G1 = add_elevation_and_slope(G1, raster_path)

##Steigung Bewertung

#Umwandlung in GeoDataFrame
edges = ox.graph_to_gdfs(G1, nodes=False)
edges = edges[edges['grade'] >= 0]
#edges['grade'] = edges['grade'].apply(lambda x: max(x, 0)) # Negative Steigung = 0
#Umwandlung in Prozent 
edges['grade'] = edges['grade'] * 100
#Funktion zur Bewertung der Steigung (grade)
def grade_score(grade):
    if grade > 20:
        return 0
    elif 10 < grade <= 20:
        return 1
    elif 7 < grade <= 10:
        return 2
    elif 5 < grade <= 7:
        return 3
    elif 3 < grade <= 5:
        return 4
    elif 2 < grade <= 3:
        return 5
    elif 1 < grade <= 2:
        return 6
    elif 0.5 < grade <= 1:
        return 7
    elif 0.2 < grade <= 0.5:
        return 8
    elif 0 < grade <= 0.2:
        return 9
    else:  # grade == 0
        return 10
    
# Anwenden der Bewertungsfunktionen auf die Daten  
edges['grade_score'] = edges['grade'].apply(grade_score)
# Berechnen der gewichteten Scores (Steigung)
edges['weighted_grade_score'] = edges['grade_score'] * edges['length']
# Berechnen des Gesamtwerts der gewichteten Scores
total_weighted_grade_score = edges['weighted_grade_score'].sum()
# Berechnen der Gesamtlänge aller Kanten
total_length = edges['length'].sum()
# Berechnen des durchschnittlichen gewichteten 'grade_score'
slope_score = total_weighted_grade_score / total_length

print(f'Gewichteter Slope_Score: {slope_score}')


ModuleNotFoundError: No module named 'osgeo'

Bicycle Separation (Segment)

In [2]:


# Funktion zur Bewertung der Straßen- und Radwegtypen unter Berücksichtigung fehlender Spalten
def highway_cycleway_score(row):
    scores = []
    if 'highway' in row and row['highway'] == 'cycleway':
        scores.append(10)
    if 'bicycle' in row and row['bicycle'] == 'designated':
        scores.append(10)
    if 'cycleway' in row:
        if row['cycleway'] == 'track' or row['cycleway'] == 'opposite_track':
            scores.append(10)
        if row['cycleway'] == 'lane' or row['cycleway'] == 'opposite_lane':
            scores.append(5)
    if 'cycleway:right' in row:
        if row['cycleway:right'] == 'track' or row['cycleway:right'] == 'opposite_track':
            scores.append(10)
        if row['cycleway:right'] == 'lane':
            scores.append(5)
    if 'cycleway:left' in row:
        if row['cycleway:left'] == 'track' or row['cycleway:left'] == 'opposite_track':
            scores.append(10)
        if row['cycleway:left'] == 'lane':
            scores.append(5)
    if 'cycleway:both' in row and row['cycleway:both'] == 'lane':
        scores.append(5)
    if 'bicycle_road' in row and row['bicycle_road'] == 'yes':
        scores.append(5)

    return max(scores) if scores else 3.33 # 0 Punkte, falls keine der Bedingungen zutrifft


edges['highway_cycleway_score'] = edges.apply(highway_cycleway_score, axis=1)

# Berechnen der gewichteten Scores (Separation)
edges['weighted_highway_cycleway_score'] = edges['highway_cycleway_score'] * edges['length']
# Berechnen des Gesamtwerts der gewichteten Scores
total_weighted_highway_cycleway_score = edges['weighted_highway_cycleway_score'].sum()
# Berechnen der Gesamtlänge aller Kanten
total_length = edges['length'].sum()
# Berechnen des durchschnittlichen gewichteten 'grade_score'
separation_score = total_weighted_highway_cycleway_score / total_length

print(f'Gewichteter Separation_Score: {separation_score}')

NameError: name 'edges' is not defined

Surface (Segment)

In [3]:
import osmnx as ox
import math
import pandas as pd
import numpy as np
from osgeo import gdal
# OSMnx konfigurieren
ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing:markings',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

latitude, longitude =53.12465745582293, 8.183655789297264#51.58086890059515,13.738319203449915#48.09390019797982, 11.194383409473405#48.80520352412432,9.169831239419484#

 #

meters_per_lat = 111320
meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
delta_lat = 5000/ meters_per_lat
delta_lon = 5000/ meters_per_lon
north = latitude + delta_lat
south = latitude
east = longitude + delta_lon
west = longitude
bbox = (west, south, east, north)
# Bewertung der Oberfläche (surface)

G1 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

# Entfernen von "service" Kanten
service_edges = [(u, v, k) for u, v, k, d in G1.edges(keys=True, data=True) if d.get('highway') == 'service']
G1.remove_edges_from(service_edges)

edges = ox.graph_to_gdfs(G1, nodes=False)
surface_scores = {
    'asphalt': 10,'concrete': 10,
    'concrete:lanes': 6,'concrete:plates': 6, 'plates': 6, 'paving_stones': 6,
    'paved': 5,
    'compacted': 4, 'dirt': 4, 'unpaved': 4, 'ground' : 4, 'sett': 4, 'metal': 4, 'wood': 4, 'gravel': 4, 'grass': 4,
    'unhewn_cobblestone': 2, 'cobblestone': 2, 'sand': 2,
}

# Funktion zur Bewertung der Oberfläche
def surface_score(surface):
    # Prüfen, ob der Wert eine Liste oder ein Array ist
    if isinstance(surface, list) or isinstance(surface, np.ndarray):
        # Nehmen Sie den ersten Wert der Liste/Array oder geben Sie 0 zurück, wenn die Liste leer ist
        surface_value = surface[0] if surface else None
    else:
        surface_value = surface

    # Standardwert auf 0 setzen, falls die Oberfläche nicht bekannt ist oder fehlt
    return surface_scores.get(surface_value, None)

edges['surface_score'] = edges['surface'].apply(surface_score)


edges = edges.dropna(subset = ['surface_score'])
# Berechnen der gewichteten Scores (Surface)
edges['weighted_surface_score'] = edges['surface_score'] * edges['length']
# Berechnen des Gesamtwerts der gewichteten Scores
total_weighted_surface_score = edges['weighted_surface_score'].sum()
# Berechnen der Gesamtlänge aller Kanten
total_length = edges['length'].sum()
# Berechnen des durchschnittlichen gewichteten 'grade_score'
surfaces_score = total_weighted_surface_score / total_length

print(f'Gewichteter Surfaces_Score: {surfaces_score}')


ModuleNotFoundError: No module named 'osgeo'

Road Class Speed & Traffic (Segment)

In [ ]:
road_scores = {
    'trunk': 0, 'trunk_link': 0,
    'primary': 1, 'primary_link': 1,
    'secondary': 3,'secondary_link': 3,
    'tertiary': 4, 'tertiary_link': 4,
    'secondary': 5,'secondary_link': 5,
    'residential': 7, 'living_street': 7,
    'path': 8, 'track': 8, 'pedestrian': 8,
    'cycleway': 10,
}

# Funktion zur Bewertung der Oberfläche
def road_score(highway):
    # Prüfen, ob der Wert eine Liste oder ein Array ist
    if isinstance(highway, list) or isinstance(highway, np.ndarray):
        # Nehmen Sie den ersten Wert der Liste/Array oder geben Sie 0 zurück, wenn die Liste leer ist
        road_scores_value = highway[0] if highway else None
    else:
        road_scores_value = highway
    return road_scores.get(road_scores_value, None)

edges['road_score'] = edges['highway'].apply(road_score)

edges = edges.dropna(subset = ['road_score'])
# Berechnen der gewichteten Scores (Road_class)
edges['weighted_road_score'] = edges['road_score'] * edges['length']
# Berechnen des Gesamtwerts der gewichteten Scores
total_weighted_road_score = edges['weighted_road_score'].sum()
# Berechnen der Gesamtlänge aller Kanten
total_length = edges['length'].sum()
# Berechnen des durchschnittlichen gewichteten 'grade_score'
road_class_score = total_weighted_road_score / total_length

print(f'Gewichteter Road_Class_Score: {road_class_score}')

Infrastructure with Separated Lanes & Road Class

In [ ]:
import pandas as pd
import numpy as np

## Neue Road Class/Separation Gewichtung

# Definition der road_scores
road_scores = {
    'trunk': 0, 'trunk_link': 0,
    'primary': 1, 'primary_link': 1,
    'secondary': 3, 'secondary_link': 3,
    'tertiary': 4, 'tertiary_link': 4,
    'residential': 7, 'living_street': 7,
    'path': 8, 'track': 8, 'pedestrian': 8,
    'cycleway': 10,
}

def calculate_adjusted_road_score(row):
    # Prüfen, ob der 'highway'-Wert eine Liste ist, und den ersten Wert verwenden, falls ja
    highway_value = row['highway'][0] if isinstance(row['highway'], list) else row['highway']
    
    # Basis-Score basierend auf der Straßenklassifizierung
    base_score = road_scores.get(highway_value, 0)
    
    # Anpassung der Punktezahl basierend auf Fahrradwegbedingungen
    multiplier = 1  # Standardmultiplikator ist 1
    if highway_value == 'cycleway' or \
       row.get('bicycle') == 'designated' or \
       (row.get('cycleway') in ['track', 'opposite_track']) or \
       (row.get('cycleway:right') in ['track', 'opposite_track']) or \
       (row.get('cycleway:left') in ['track', 'opposite_track']):
        multiplier = 3
    elif row.get('bicycle_road') == 'yes' or \
         row.get('cycleway:both') == 'lane' or \
         row.get('cycleway') in ['lane', 'opposite_lane'] or \
         row.get('cycleway:right') == 'lane' or \
         row.get('cycleway:left') == 'lane':
        multiplier = 2
    
    # Angepassten Score berechnen
    adjusted_score = base_score * multiplier
    return adjusted_score

# Berechnen des angepassten Scores für jede Kante
edges['adjusted_road_score'] = edges.apply(calculate_adjusted_road_score, axis=1)

# Berechnen der gewichteten Scores
edges['weighted_road_score'] = edges['adjusted_road_score'] * edges['length']

# Berechnen des Gesamtwerts der gewichteten Scores und der Gesamtlänge
total_weighted_road_score = edges['weighted_road_score'].sum()
total_length = edges['length'].sum()

# Durchschnittlichen gewichteten Score berechnen
road_class_score = total_weighted_road_score / total_length

print(f'Gewichteter Road_Class_Score mit Anpassungen für Radwege: {road_class_score}')


Gitterbewertung

In [ ]:
G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)


##Fahrradroutenlänge


non_cyc2 = []
for u, v, k, d in G6.edges(keys=True, data=True):
    if d.get('bicycle') == 'separate' or d.get('cycleway') == 'separate' \
       or d.get('cycleway:right') == 'separate' or d.get('cycleway:left') == 'separate' \
       or d.get('cycleway:both') == 'separate':
        non_cyc2.append((u, v, k))
    if d.get('bicycle') in ['designated','use_sidepath']:
        continue
    elif d.get('highway') in ['cycleway'] :
        continue    
    elif d.get('cycleway') in ['track']:
        continue
    elif d.get('cycleway:right') in ['track']:
        continue
    elif d.get('cycleway:left') in ['track']:
        continue
    elif d.get('cycleway:both') in ['track']:
        continue
    non_cyc2.append((u, v, k))
G6.remove_edges_from(non_cyc2)
G6 = ox.utils_graph.remove_isolated_nodes(G6)
#fig, ax = ox.plot_graph(G6)
stats2 = ox.stats.basic_stats(G6)
streetlength = stats2['street_length_total']
print(streetlength)

scale_factor = 49.74
scaled_ranges = {
    1: (round(0*scale_factor),round(0*scale_factor)),
    2: (round(1 * scale_factor), round(250 * scale_factor)),  # 1-12435 m (hochskaliert)
    3: (round(251 * scale_factor), round(450 * scale_factor)),# 12486-22383 m (hochskaliert)
    4: (round(451 * scale_factor), round(600 * scale_factor)),# 22434-29844 m (hochskaliert)
    5: (round(601 * scale_factor), round(750 * scale_factor)),# 29895-37305 m (hochskaliert)
    6: (round(751 * scale_factor), round(850 * scale_factor)),# 37356-42273 m (hochskaliert)
    7: (round(851 * scale_factor), round(1100 * scale_factor)),# 42324-54742 m (hochskaliert)
    8: (round(1101 * scale_factor), round(1400 * scale_factor)),# 54793-69628 m (hochskaliert)
    9: (round(1401 * scale_factor), round(1800 * scale_factor)),# 69679-89508 m (hochskaliert)
    10: (round(1801 * scale_factor), round(10000 * scale_factor))# 89559-298440 m (hochskaliert)
}

# Ermittlung der Bewertung
# Standardwert für Werte über dem höchsten Bereich

for key, (low, high) in scaled_ranges.items():
    if low <= streetlength <= high:
        fahrradroute_score = key
        break
print("Länge der Fahrradroute:", streetlength)
print("Fahrradroute_Score:", fahrradroute_score)


##Connectivity


inters = ox.stats.intersection_count(G6, min_streets = 2)

scale_factor = 49.74
scaled_ranges = {
1: (round(0 * scale_factor), round(0 * scale_factor)),            # 0 Kreuzungen (hochskaliert)
2: (round(1), round(1 * scale_factor)),                           # 1-49 Kreuzungen (hochskaliert)
3: (round(1 * scale_factor + 1), round(3 * scale_factor)),       # 50-149 Kreuzungen (hochskaliert)
4: (round(3 * scale_factor + 1), round(6 * scale_factor)),       # 150-298 Kreuzungen (hochskaliert)
5: (round(6 * scale_factor + 1), round(10 * scale_factor)),      # 299-497 Kreuzungen (hochskaliert)
6: (round(10 * scale_factor + 1), round(15 * scale_factor)),     # 498-746 Kreuzungen (hochskaliert)
7: (round(15 * scale_factor + 1), round(20 * scale_factor)),     # 747-994 Kreuzungen (hochskaliert)
8: (round(20 * scale_factor + 1), round(25 * scale_factor)),     # 995-1243 Kreuzungen (hochskaliert)
9: (round(25 * scale_factor + 1), round(30 * scale_factor)),     # 1244-1492 Kreuzungen (hochskaliert)
10: (round(30 * scale_factor + 1), round(100 * scale_factor))    # 1493-4974 Kreuzungen (hochskaliert)

}
# Ermittlung der Bewertung
# Standardwert für Werte über dem höchsten Bereich
for key, (low, high) in scaled_ranges.items():
    if low <= inters <= high:
        connectivity_score = key
        break

print("Anzahl der Kreuzungen:", inters)
print("Connectivity_Score:", connectivity_score)


##Main_Roads


G7 = ox.graph_from_bbox(north, south, east, west, network_type='drive', simplify=True, retain_all=True, truncate_by_edge=True)

# Liste der zu entfernenden Kanten
edges_to_remove = []
for u, v, key, data in G7.edges(keys=True, data=True):
    if data.get('bicycle') == 'separate' or \
    data.get('cycleway') == 'separate' or \
    data.get('cycleway:right') == 'separate' or \
    data.get('cycleway:left') == 'separate' or \
    data.get('cycleway:both') == 'separate' or \
    data.get('bicycle') in ['designated','yes' 'use_sidepath'] or \
    data.get('highway') in ['residential', 'cycleway', 'pedestrian', 'track', 'raceway', 'living_street']  or \
    data.get('cycleway') in ['track', 'lane', 'shared_lane'] or \
    data.get('cycleway:right') in ['track', 'lane', 'shared_lane'] or \
    data.get('cycleway:left') in ['track', 'lane', 'shared_lane'] or \
    data.get('cycleway:both') in ['track', 'lane', 'shared_lane']:
        edges_to_remove.append((u, v, key))

# Entfernen der Kanten aus dem Graphen
G7.remove_edges_from(edges_to_remove)

# Berechnen der Gesamtlänge der Hauptstraßen
stats2 = ox.stats.basic_stats(G7)
length = stats2['street_length_total']


# Konvertieren des Graphen zu GeoDataFrame
edges7 = ox.graph_to_gdfs(G7, nodes=False)
scale_factor = 127.324
scaled_ranges = {
    1: (round(1101 * scale_factor +1), round(3565 * scale_factor)), 
    2: (round(883 * scale_factor +1), round(1101 * scale_factor)),                                # 1-31957 m (hochskaliert)
    3: (round(726 * scale_factor + 1), round(883 * scale_factor)),           # 31958-54753 m (hochskaliert)
    4: (round(585 * scale_factor + 1), round(726 * scale_factor)),           # 54754-69371 m (hochskaliert)
    5: (round(491 * scale_factor + 1), round(585 * scale_factor)),           # 69372-89356 m (hochskaliert)
    6: (round(405 * scale_factor + 1), round(491 * scale_factor)),           # 89357-111314 m (hochskaliert)
    7: (round(288 * scale_factor + 1), round(405 * scale_factor)),          # 111315-135355 m (hochskaliert)
    8: (round(160 * scale_factor + 1), round(288 * scale_factor)),         # 135356-167219 m (hochskaliert)
    9: (round(1), round(160 * scale_factor)),         # 167220-209380 m (hochskaliert)
    10: (round(0), round(0))         
}

# Ermittlung der Bewertung
# Standardwert für Werte über dem höchsten Bereich

for key, (low, high) in scaled_ranges.items():
    if low <= length <= high:
        main_roads_score = key
        break
print(f"Main Roads Length: {length}")
print(f"Main Roads Length Score: {main_roads_score}")

Fahrradstraßenlänge (noch nicht in Relation) (Fahrradroute_Score)

In [ ]:
G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)
service_edges = [(u, v, k) for u, v, k, d in G6.edges(keys=True, data=True) if d.get('highway') == 'service']

non_cyc2 = []
for u, v, k, d in G6.edges(keys=True, data=True):
    if d.get('bicycle') == 'separate' or d.get('cycleway') == 'separate' \
       or d.get('cycleway:right') == 'separate' or d.get('cycleway:left') == 'separate' \
       or d.get('cycleway:both') == 'separate':
        non_cyc2.append((u, v, k))
    if d.get('bicycle') in ['designated','use_sidepath']:
        continue
    elif d.get('highway') in ['cycleway'] :
        continue    
    elif d.get('cycleway') in ['track']:
        continue
    elif d.get('cycleway:right') in ['track']:
        continue
    elif d.get('cycleway:left') in ['track']:
        continue
    elif d.get('cycleway:both') in ['track']:
        continue
    non_cyc2.append((u, v, k))
G6.remove_edges_from(non_cyc2)
G6 = ox.utils_graph.remove_isolated_nodes(G6)
#fig, ax = ox.plot_graph(G6)
streetlength = ox.stats.edge_length_total(G6)
print(streetlength)

scale_factor = 49.74
scaled_ranges = {
    1: (round(0*scale_factor),round(0*scale_factor)),
    2: (round(1 * scale_factor), round(250 * scale_factor)),  # 1-12435 m (hochskaliert)
    3: (round(251 * scale_factor), round(450 * scale_factor)),# 12486-22383 m (hochskaliert)
    4: (round(451 * scale_factor), round(600 * scale_factor)),# 22434-29844 m (hochskaliert)
    5: (round(601 * scale_factor), round(750 * scale_factor)),# 29895-37305 m (hochskaliert)
    6: (round(751 * scale_factor), round(850 * scale_factor)),# 37356-42273 m (hochskaliert)
    7: (round(851 * scale_factor), round(1100 * scale_factor)),# 42324-54742 m (hochskaliert)
    8: (round(1101 * scale_factor), round(1400 * scale_factor)),# 54793-69628 m (hochskaliert)
    9: (round(1401 * scale_factor), round(1800 * scale_factor)),# 69679-89508 m (hochskaliert)
    10: (round(1801 * scale_factor), round(10000 * scale_factor))# 89559-298440 m (hochskaliert)
}

# Ermittlung der Bewertung
# Standardwert für Werte über dem höchsten Bereich

for key, (low, high) in scaled_ranges.items():
    if low <= streetlength <= high:
        fahrradroute_score = key
        break
print("Länge der Fahrradroute:", streetlength)
print("Bewertungspunkt:", fahrradroute_score)


Connectivity Kreuzung (Connectivity_Score)

In [36]:
import osmnx as ox
import math
import pandas as pd
import numpy as np
from osgeo import gdal
# OSMnx konfigurieren
ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing:markings',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

latitude, longitude =(#53.12465745582293, 8.183655789297264
 #48.09390019797982,11.194383409473405
 #48.80520352412432,9.169831239419484
)
meters_per_lat = 111320
meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
delta_lat = 5000/ meters_per_lat
delta_lon = 5000/ meters_per_lon
north = latitude + delta_lat
south = latitude
east = longitude + delta_lon
west = longitude
bbox = (west, south, east, north)


G5 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

non_cyc = []
for u, v, k, d in G5.edges(keys=True, data=True):
    if d.get('bicycle') == 'separate' or d.get('cycleway') == 'separate' \
    or d.get('cycleway:right') == 'separate' or d.get('cycleway:left') == 'separate' \
    or d.get('cycleway:both') == 'separate':
        non_cyc.append((u, v, k))

    if d.get('bicycle') in ['designated','use_sidepath']:
        continue
    elif d.get('highway') in ['cycleway'] :
        continue    
    elif d.get('cycleway') in ['track', 'opposite_track']:
        continue
    elif d.get('cycleway:right') in ['track', 'opposite_track']:
        continue
    elif d.get('cycleway:left') in ['track', 'opposite_track']:
        continue
    elif d.get('cycleway:both') in ['track', 'opposite_track']:
        continue   
    elif d.get('highway') == 'path' and d.get('bicycle') == 'yes':
        continue
    non_cyc.append((u, v, k))
G5.remove_edges_from(non_cyc)
G5 = ox.utils_graph.remove_isolated_nodes(G5)
#fig, ax = ox.plot_graph(G5)
inters = ox.stats.intersection_count(G5, min_streets = 3)

scale_factor = 49.74
scaled_ranges = {
1: (round(0 * scale_factor), round(0 * scale_factor)),            # 0 Kreuzungen (hochskaliert)
2: (round(1), round(1 * scale_factor)),                           # 1-49 Kreuzungen (hochskaliert)
3: (round(1 * scale_factor + 1), round(3 * scale_factor)),       # 50-149 Kreuzungen (hochskaliert)
4: (round(3 * scale_factor + 1), round(6 * scale_factor)),       # 150-298 Kreuzungen (hochskaliert)
5: (round(6 * scale_factor + 1), round(10 * scale_factor)),      # 299-497 Kreuzungen (hochskaliert)
6: (round(10 * scale_factor + 1), round(15 * scale_factor)),     # 498-746 Kreuzungen (hochskaliert)
7: (round(15 * scale_factor + 1), round(20 * scale_factor)),     # 747-994 Kreuzungen (hochskaliert)
8: (round(20 * scale_factor + 1), round(25 * scale_factor)),     # 995-1243 Kreuzungen (hochskaliert)
9: (round(25 * scale_factor + 1), round(30 * scale_factor)),     # 1244-1492 Kreuzungen (hochskaliert)
10: (round(30 * scale_factor + 1), round(100 * scale_factor))    # 1493-4974 Kreuzungen (hochskaliert)

}
# Ermittlung der Bewertung
# Standardwert für Werte über dem höchsten Bereich
for key, (low, high) in scaled_ranges.items():
    if low <= inters <= high:
        connectivity_score = key
        break

print("Anzahl der Kreuzungen:", inters)
print("Bewertungspunkt:", connectivity_score)

C:\Users\kevdr\AppData\Local\Temp\ipykernel_20428\3927864613.py:7: UserWarning: The `utils.config` function is deprecated and will be removed in a future release. Instead, use the `settings` module directly to configure a global setting's value. For example, `ox.settings.log_console=True`.
  ox.config(use_cache=True, log_console=True)


Anzahl der Kreuzungen: 1536
Bewertungspunkt: 10


Hauptstraßenlänge (Main_Roads_Score)

In [ ]:
G7 = ox.graph_from_bbox(north, south, east, west, network_type='drive', simplify=True, retain_all=True, truncate_by_edge=True)

# Liste der zu entfernenden Kanten
edges_to_remove = []
for u, v, key, data in G7.edges(keys=True, data=True):
    if data.get('bicycle') == 'separate' or \
    data.get('cycleway') == 'separate' or \
    data.get('cycleway:right') == 'separate' or \
    data.get('cycleway:left') == 'separate' or \
    data.get('cycleway:both') == 'separate' or \
    data.get('bicycle') in ['designated','yes' 'use_sidepath'] or \
    data.get('highway') in ['residential', 'cycleway', 'pedestrian', 'track', 'raceway', 'living_street']  or \
    data.get('cycleway') in ['track', 'lane', 'shared_lane'] or \
    data.get('cycleway:right') in ['track', 'lane', 'shared_lane'] or \
    data.get('cycleway:left') in ['track', 'lane', 'shared_lane'] or \
    data.get('cycleway:both') in ['track', 'lane', 'shared_lane']:
        edges_to_remove.append((u, v, key))

# Entfernen der Kanten aus dem Graphen
G7.remove_edges_from(edges_to_remove)

# Berechnen der Gesamtlänge der Hauptstraßen
length = ox.stats.edge_length_total(G7)


# Konvertieren des Graphen zu GeoDataFrame
edges7 = ox.graph_to_gdfs(G7, nodes=False)
scale_factor = 127.324
scaled_ranges = {
    1: (round(1101 * scale_factor +1), round(3565 * scale_factor)), 
    2: (round(883 * scale_factor +1), round(1101 * scale_factor)),                                # 1-31957 m (hochskaliert)
    3: (round(726 * scale_factor + 1), round(883 * scale_factor)),           # 31958-54753 m (hochskaliert)
    4: (round(585 * scale_factor + 1), round(726 * scale_factor)),           # 54754-69371 m (hochskaliert)
    5: (round(491 * scale_factor + 1), round(585 * scale_factor)),           # 69372-89356 m (hochskaliert)
    6: (round(405 * scale_factor + 1), round(491 * scale_factor)),           # 89357-111314 m (hochskaliert)
    7: (round(288 * scale_factor + 1), round(405 * scale_factor)),          # 111315-135355 m (hochskaliert)
    8: (round(160 * scale_factor + 1), round(288 * scale_factor)),         # 135356-167219 m (hochskaliert)
    9: (round(1), round(160 * scale_factor)),         # 167220-209380 m (hochskaliert)
    10: (round(0), round(0))         
}

# Ermittlung der Bewertung
# Standardwert für Werte über dem höchsten Bereich

for key, (low, high) in scaled_ranges.items():
    if low <= length <= high:
        main_roads_score = key
        break
print(f"Main Roads Length: {length}")
print(f"Main Roads Length Score: {main_roads_score}")

Anteil Grünfläche

In [82]:
import math
import pandas as pd
import numpy as np
from osgeo import gdal
import osmnx as ox
import geopandas as gpd
from shapely.geometry import box


# OSMnx konfigurieren
ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing:markings',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

latitude, longitude =47.80359124161864,8.18530756879351
#49.11999564530412,9.164533737241891

 #48.80520352412432,9.169831239419484

meters_per_lat = 111320
meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
delta_lat = 5000/ meters_per_lat
delta_lon = 5000/ meters_per_lon
north = latitude + delta_lat
south = latitude
east = longitude + delta_lon
west = longitude
bbox = (west, south, east, north)

# Funktion zur Bewertung des Grünflächenanteils
def bewerte_gruenflaechen(anteil):
    if anteil == 0:
        return 0
    elif 9 < anteil <= 22:
        return 2
    elif 22 < anteil <= 41:
        return 4
    elif 41 < anteil <= 61:
        return 6
    elif 61 < anteil <= 81:
        return 8
    elif 81 < anteil:
        return 10
    else:
        return "Anteil außerhalb des definierten Bereichs"
#tags = {}
#tags = {'landuse': 'forest'}
tags = {'leisure': 'park', 'landuse': ['grass','forest'], 'natural': ['water','scrub', 'wetland', 'coastline', 'sand']}

# Extrahieren von Grünflächen innerhalb der Bounding Box mit OSMnx
green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)

# Erstellen eines GeoDataFrame für die Grünflächen
green_spaces_gdf = gpd.GeoDataFrame(green_spaces, geometry='geometry', crs="EPSG:4326")

# Transformieren in die flächentreue Projektion EPSG:3035 für die Flächenberechnung
green_spaces_gdf = green_spaces_gdf.to_crs("EPSG:3035")

# Berechnen der Gesamtfläche der Grünflächen in Quadratkilometern
total_green_area = green_spaces_gdf.area.sum() / 10**6

# Erstellen eines Polygons für die Bounding Box und Berechnen der Gesamtfläche
bbox_polygon = box(west, south, east, north)
bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox_polygon]}, crs="EPSG:4326").to_crs("EPSG:3035")
bbox_area_km2 = bbox_gdf.area.sum() / 10**6

# Berechnen des prozentualen Anteils der Grünflächen
percent_green_space = (total_green_area/ bbox_area_km2) * 100


# Anwenden des Bewertungssystems
green_score = bewerte_gruenflaechen(percent_green_space)

print(f"Grünfläche in km²: {total_green_area:.2f}")
print(f"Anteil der Grünfläche: {percent_green_space:.2f}%")
print(f"Punkte für Grünflächenanteil: {green_score}")


C:\Users\kevdr\AppData\Local\Temp\ipykernel_24952\3079810236.py:11: UserWarning: The `utils.config` function is deprecated and will be removed in a future release. Instead, use the `settings` module directly to configure a global setting's value. For example, `ox.settings.log_console=True`.
  ox.config(use_cache=True, log_console=True)
C:\Users\kevdr\AppData\Local\Temp\ipykernel_24952\3079810236.py:56: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in km²: 104.37
Anteil der Grünfläche: 417.39%
Punkte für Grünflächenanteil: 10


Fahrradfreundliche Straßen(Relation)

In [170]:
import osmnx as ox
import math
import pandas as pd
import numpy as np
from osgeo import gdal
# OSMnx konfigurieren
ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing:markings',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

latitude, longitude =(48.9759833720223,8.347473283166716
    #47.80359124161864,8.18530756879351
    #48.97495362005015,8.279208890041579
    #47.976121765929626,7.7773975938558655
    #48.09390019797982, 11.194383409473405
    #47.976121765929626,7.7773975938558655#,
#48.09390019797982, 11.194383409473405 #Pampa um München'
#53.12465745582293, 8.183655789297264 #Oldenburg'
#51.93670073154713, 7.598923696853611 #Münster
#52.493468042048725, 13.368426807546985 #Berlin
#51.58086890059515,13.738319203449915#,Dorf im Norden
)

meters_per_lat = 111320
meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
delta_lat = 5000/ meters_per_lat
delta_lon = 5000/ meters_per_lon
north = latitude + delta_lat
south = latitude
east = longitude + delta_lon
west = longitude
bbox = (west, south, east, north)

G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

service_edges = [(u, v, k) for u, v, k, d in G6.edges(keys=True, data=True) if d.get('highway') == 'service']
G6.remove_edges_from(service_edges)
G6 = ox.utils_graph.remove_isolated_nodes(G6)

alledges = ox.graph_to_gdfs(G6, nodes=False)
total_length_all = alledges['length'].sum()


G7 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

non_cyc2 =[]
for u, v, k, d in G7.edges(keys=True, data=True):
    if d.get('highway') in ['cycleway', 'residential', 'tertiary','tertiary_link', 'living_street','unclassified'] :
        continue 
    elif d.get('cycleway') in ['track', 'opposite_track']:
        continue
    elif d.get('cycleway:right') in ['track', 'opposite_track']:
        continue
    elif d.get('cycleway:left') in ['track', 'opposite_track']:
        continue
    elif d.get('cycleway:both') in ['track', 'opposite_track']:
        continue   
    elif d.get('bicycle') in ['designated','use_sidepath']:
        continue
    elif d.get('highway') == 'track'  and d.get('tracktype') in ['grade1','grade2']:
        continue   
    elif d.get('highway') == 'path' and d.get('bicycle') == 'yes':
        continue
    elif d.get('bicycle_road') == 'yes':
        continue
    non_cyc2.append((u, v, k))

G7.remove_edges_from(non_cyc2+ service_edges)
G7 = ox.utils_graph.remove_isolated_nodes(G7)
bikefriendlyedges = ox.graph_to_gdfs(G7, nodes=False)

total_length = bikefriendlyedges['length'].sum()
# Verwenden Sie eine sicherere Methode, um die angepasste Länge zu berechnen
def calculate_adjusted_length(row):
    # Prüfen, ob die notwendigen Spalten existieren und die Bedingungen erfüllen
    conditions = [
        row.get('highway') == 'cycleway',
    ]
    # Verdoppeln Sie die Länge, wenn eine der Bedingungen erfüllt ist
    if any(conditions):
        return row['length'] * 2
    else:
        return row['length']

# Berechnen Sie die angepasste Länge für jede Zeile
bikefriendlyedges['adjusted_length'] = bikefriendlyedges.apply(calculate_adjusted_length, axis=1)


adjusted_total_length = bikefriendlyedges['adjusted_length'].sum()

bike_friendly_street_share = adjusted_total_length/total_length_all * 100


def bewerte_fahrradfreund(anteil):
    if anteil == 0:
        return 0
    elif 0 < anteil <= 26:
        return 2
    elif 26 < anteil <= 58:
        return 4
    elif 58 < anteil <= 75:
        return 6
    elif 75 < anteil <= 87:
        return 8
    elif 87 < anteil:
        return 10
    else:
        return "Anteil außerhalb des definierten Bereichs"
# Ermittlung der Bewertung
# Standardwert für Werte über dem höchsten Bereich

fahrradroute_score = bewerte_fahrradfreund(bike_friendly_street_share)

print("Länge der Fahrradroute ohne Gewichtung:", total_length )
print("Länge der Fahrradroute:", adjusted_total_length)
print("Länge Insgesamt:", total_length_all)
print("Prozentualer Share:", bike_friendly_street_share)
print("Bewertungspunkt:", fahrradroute_score)


C:\Users\kevdr\AppData\Local\Temp\ipykernel_24952\4202265814.py:7: UserWarning: The `utils.config` function is deprecated and will be removed in a future release. Instead, use the `settings` module directly to configure a global setting's value. For example, `ox.settings.log_console=True`.
  ox.config(use_cache=True, log_console=True)


Länge der Fahrradroute ohne Gewichtung: 572032.296
Länge der Fahrradroute: 611621.206
Länge Insgesamt: 866992.5349999999
Prozentualer Share: 70.54515250237998
Bewertungspunkt: 6


In [11]:
bbox

(12.970492, 54.372572, 13.047599032913299, 54.41748755874955)

Haupstraßeninfrastruktur

In [22]:


import osmnx as ox
import pandas as pd

import osmnx as ox
import math
import pandas as pd
import numpy as np
from osgeo import gdal
# OSMnx konfigurieren
ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing:markings',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

latitude, longitude =(54.339406,8.606838
#47.976121765929626,7.7773975938558655
#48.09390019797982, 11.194383409473405 #Pampa um München'
#53.12465745582293, 8.183655789297264 #Oldenburg'
#51.93670073154713, 7.598923696853611 #Münster
#52.493468042048725, 13.368426807546985 #Berlin
#51.58086890059515,13.738319203449915#,Dorf im Norden
)

meters_per_lat = 111320
meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
delta_lat = 5000/ meters_per_lat
delta_lon = 5000/ meters_per_lon
north = latitude + delta_lat
south = latitude
east = longitude + delta_lon
west = longitude
bbox = (west, south, east, north)

graph = ox.graph_from_bbox(north, south, east, west, network_type='bike', custom_filter='["highway"~"primary|primary_link|secondary|secondary_link|trunk|trunk_link"]', simplify=False, retain_all=True, truncate_by_edge=True)

# Konvertiere das Graphenobjekt in ein GeoDataFrame
edges = ox.graph_to_gdfs(graph, nodes=False)
# Zuerst erstellen Sie eine Funktion, die prüft, ob die Spalte existiert und den Filter anwendet
def check_and_filter(df, column, value):
    if column in df.columns:
        return df[column] == value
    else:
        return pd.Series([False] * len(df), index=df.index)

# Dann passen Sie Ihre Filterlogik an, um diese Funktion zu nutzen
bike_infrastructure_filter = (
    check_and_filter(edges, 'cycleway', 'lane') |
    check_and_filter(edges, 'bicycle', 'designated') |
    check_and_filter(edges, 'cycleway', 'track') |
    check_and_filter(edges, 'cycleway:right', 'track') |
    check_and_filter(edges, 'cycleway:left', 'track') |
    check_and_filter(edges, 'cycleway:both', 'track') |
    check_and_filter(edges, 'cycleway:right', 'lane') |
    check_and_filter(edges, 'cycleway:left', 'lane') |
    check_and_filter(edges, 'cycleway:both', 'lane') |
    check_and_filter(edges, 'bicycle', 'use_sidepath') |
    check_and_filter(edges, 'cycleway', 'separate') |
    check_and_filter(edges, 'cycleway:right', 'separate') |
    check_and_filter(edges, 'cycleway:left', 'separate') |
    check_and_filter(edges, 'cycleway:both', 'separate') 
)

bike_lanes = edges[bike_infrastructure_filter]

# Berechnen Sie die Länge der Fahrradspuren und führen Sie die weiteren Berechnungen wie zuvor durch
# Summiere die Gesamtlänge und die angepasste Länge
total_length = bike_lanes['length'].sum()

# Verwenden Sie eine sicherere Methode, um die angepasste Länge zu berechnen
def calculate_adjusted_length(row):
    # Prüfen, ob die notwendigen Spalten existieren und die Bedingungen erfüllen
    conditions = [
        row.get('cycleway') == 'track',
        row.get('bicycle') == 'designated',
        row.get('cycleway:right') == 'track',
        row.get('cycleway:left') == 'track',
        row.get('cycleway:both') == 'track',
        row.get('bicycle') == 'use_sidepath'
    ]
    # Verdoppeln Sie die Länge, wenn eine der Bedingungen erfüllt ist
    if any(conditions):
        return row['length'] 
    else:
        return row['length']

# Berechnen Sie die angepasste Länge für jede Zeile
bike_lanes['adjusted_length'] = bike_lanes.apply(calculate_adjusted_length, axis=1)


adjusted_total_length = bike_lanes['adjusted_length'].sum()

# Berechnen Sie die Gesamtlänge aller primären und sekundären Straßen
all_edges_length = edges['length'].sum()

# Ausgabe der Ergebnisse
print(f"Gesamtlänge der Fahrradinfrastruktur: {total_length} Meter")
print(f"Angepasste Gesamtlänge unter Berücksichtigung separater Wege: {adjusted_total_length} Meter")
print(f"Gesamtlänge aller primären und sekundären Straßen: {all_edges_length} Meter")

# Berechnung des Verhältnisses
ratio = adjusted_total_length / all_edges_length * 100
print(f"Verhältnis: {ratio}%")
edges.to_csv('probe.csv')

C:\Users\kevdr\AppData\Local\Temp\ipykernel_17468\677073695.py:10: UserWarning: The `utils.config` function is deprecated and will be removed in a future release. Instead, use the `settings` module directly to configure a global setting's value. For example, `ox.settings.log_console=True`.
  ox.config(use_cache=True, log_console=True)


InsufficientResponseError: No data elements in server response. Check query location/filters and log.

Endbewertung

In [ ]:
#Endbewertung
Note = (main_roads_score + connectivity_score + fahrradroute_score + slope_score + surfaces_score + separation_score + road_class_score)/7

print(f'Gewichteter Surfaces_Score: {surfaces_score}')
print(f'Gewichteter Slope_Score: {slope_score}')
print(f'Gewichteter Road_Class_Score: {road_class_score}')
print(f'Gewichteter Separation_Score: {separation_score}')
print("Fahrradroute_Score:", fahrradroute_score)
print("Connectivity_Score:", connectivity_score)
print(f"Main Roads Length Score: {main_roads_score}")
print(f"Green_Score: {green_score}")


print(f"Endbewertung: {Note}")


Gesamtbewertungscode

In [ ]:
import osmnx as ox
import math
import pandas as pd
import numpy as np
from osgeo import gdal
# OSMnx konfigurieren
ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

file_path = 'random_coordinates.csv'
df = pd.read_csv(file_path)

# Ergebnis DataFrame vorbereiten
results = pd.DataFrame()

for index, row in df.iterrows():
    ort = row['ort']
    latitude, longitude =row['lat'], row['lon']

    #48.80520352412432,9.169831239419484

    meters_per_lat = 111320
    meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
    delta_lat = 5000/ meters_per_lat
    delta_lon = 5000/ meters_per_lon
    north = latitude + delta_lat
    south = latitude
    east = longitude + delta_lon
    west = longitude
    bbox = (west, south, east, north)


    ###Segmentbewertung


    G1 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    # Entfernen von "service" Kanten
    service_edges = [(u, v, k) for u, v, k, d in G1.edges(keys=True, data=True) if d.get('highway') == 'service']
    non_cyc3 = []
    for u, v, k, d in G1.edges(keys=True, data=True):
        if d.get('bicycle') == 'separate' or d.get('cycleway') == 'separate' \
        or d.get('cycleway:right') == 'separate' or d.get('cycleway:left') == 'separate' \
        or d.get('cycleway:both') == 'separate':
            non_cyc3.append((u, v, k))
    G1.remove_edges_from( non_cyc3  + service_edges)

    #Steigung OSMnx

    def add_elevation_and_slope(G1, raster_path):
        G1 = ox.elevation.add_node_elevations_raster(G1, raster_path)
        G1 = ox.elevation.add_edge_grades(G1, add_absolute=True)
        return G1
    raster_path = "srtm_germany_dtm.tif"

    G1 = add_elevation_and_slope(G1, raster_path)

    #Steigung Bewertung

    edges = ox.graph_to_gdfs(G1, nodes=False)
    edges['grade'] = edges['grade'].apply(lambda x: max(x, 0))
    # Anwenden der Bewertungsfunktionen auf die Daten
    edges['grade'] = edges['grade'] * 100
    # Funktion zur Bewertung der Steigung (grade)
    def grade_score(grade):
        if grade > 20:
            return 1
        elif 10 < grade <= 20:
            return 2
        elif 7 < grade <= 10:
            return 3
        elif 5 < grade <= 7:
            return 4
        elif 3 < grade <= 5:
            return 5
        elif 2 < grade <= 3:
            return 6
        elif 1 < grade <= 2:
            return 7
        elif 0.5 < grade <= 1:
            return 8
        elif 0 < grade <= 0.5:
            return 9
        else:  # grade == 0
            return 10



    # Bewertung der Oberfläche (surface)
        
    surface_scores = {
        'asphalt': 10,
        'concrete': 6, 'plates': 6, 'paving_stones': 6,
        'paved': 5,
        'sett': 4, 'metal': 4, 'wood': 4, 'gravel': 4, 'grass': 4,
        'cobblestone': 2, 'ground': 2,
    }

    # Funktion zur Bewertung der Oberfläche
    def surface_score(surface):
        # Prüfen, ob der Wert eine Liste oder ein Array ist
        if isinstance(surface, list) or isinstance(surface, np.ndarray):
            # Nehmen Sie den ersten Wert der Liste/Array oder geben Sie 0 zurück, wenn die Liste leer ist
            surface_value = surface[0] if surface else None
        else:
            surface_value = surface

        # Standardwert auf 0 setzen, falls die Oberfläche nicht bekannt ist oder fehlt
        return surface_scores.get(surface_value, None)


    # Bewertung der Straßen- und Radwegtypen

    # Funktion zur Bewertung der Straßen- und Radwegtypen unter Berücksichtigung fehlender Spalten
    def highway_cycleway_score(row):
        scores = []
        if 'highway' in row and row['highway'] == 'cycleway':
            scores.append(10)
        if 'bicycle' in row and row['bicycle'] == 'designated':
            scores.append(10)
        if 'cycleway' in row:
            if row['cycleway'] == 'track' or row['cycleway'] == 'opposite_track':
                scores.append(10)
            if row['cycleway'] == 'lane' or row['cycleway'] == 'opposite_lane':
                scores.append(5)
        if 'cycleway:right' in row:
            if row['cycleway:right'] == 'track' or row['cycleway:right'] == 'opposite_track':
                scores.append(10)
            if row['cycleway:right'] == 'lane':
                scores.append(5)
        if 'cycleway:left' in row:
            if row['cycleway:left'] == 'track' or row['cycleway:left'] == 'opposite_track':
                scores.append(10)
            if row['cycleway:left'] == 'lane':
                scores.append(5)
        if 'cycleway:both' in row and row['cycleway:both'] == 'lane':
            scores.append(5)
        if 'bicycle_road' in row and row['bicycle_road'] == 'yes':
            scores.append(5)

        return max(scores) if scores else 3.33 # 0 Punkte, falls keine der Bedingungen zutrifft

    #Bewertung Road Class

    road_scores = {
        'trunk': 0, 'trunk_link': 0,
        'primary': 1, 'primary_link': 1,
        'secondary': 3,'secondary_link': 3,
        'tertiary': 4, 'tertiary_link': 4,
        'secondary': 5,'secondary_link': 5,
        'residential': 7, 'living_street': 7,
        'path': 8, 'track': 8, 'pedestrian': 8,
        'cycleway': 10,
    }

    # Funktion zur Bewertung der Oberfläche
    def road_score(highway):
        # Prüfen, ob der Wert eine Liste oder ein Array ist
        if isinstance(highway, list) or isinstance(highway, np.ndarray):
            # Nehmen Sie den ersten Wert der Liste/Array oder geben Sie 0 zurück, wenn die Liste leer ist
            road_scores_value = highway[0] if highway else None
        else:
            road_scores_value = highway
        return road_scores.get(road_scores_value, None)

    ##Funktion anwenden
    edges['grade_score'] = edges['grade'].apply(grade_score)
    edges['surface_score'] = edges['surface'].apply(surface_score)
    edges['highway_cycleway_score'] = edges.apply(highway_cycleway_score, axis=1)
    edges['road_score'] = edges['highway'].apply(road_score)

    

    ## Gewichteter Indikatorendurchschnitt (Score)


    # Berechnen der gewichteten Scores (Steigung)
    edges['weighted_grade_score'] = edges['grade_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_grade_score = edges['weighted_grade_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    slope_score = total_weighted_grade_score / total_length

    print(f'Gewichteter Slope_Score: {slope_score}')


    edges = edges.dropna(subset = ['surface_score'])
    # Berechnen der gewichteten Scores (Surface)
    edges['weighted_surface_score'] = edges['surface_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_surface_score = edges['weighted_surface_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    surfaces_score = total_weighted_surface_score / total_length

    print(f'Gewichteter Surfaces_Score: {surfaces_score}')


    edges = edges.dropna(subset = ['road_score'])
    # Berechnen der gewichteten Scores (Road_class)
    edges['weighted_road_score'] = edges['road_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_road_score = edges['weighted_road_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    road_class_score = total_weighted_road_score / total_length

    print(f'Gewichteter Road_Class_Score: {road_class_score}')


    # Berechnen der gewichteten Scores (Separation)
    edges['weighted_highway_cycleway_score'] = edges['highway_cycleway_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_highway_cycleway_score = edges['weighted_highway_cycleway_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    separation_score = total_weighted_highway_cycleway_score / total_length

    print(f'Gewichteter Separation_Score: {separation_score}')


    ###Gitterbewertung


    G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    # Entfernen von "service" Kanten
    service_edges = [(u, v, k) for u, v, k, d in G6.edges(keys=True, data=True) if d.get('highway') == 'service']
    G6.remove_edges_from(service_edges)


    ##Fahrradroutenlänge


    non_cyc2 = []
    for u, v, k, d in G6.edges(keys=True, data=True):
        if d.get('bicycle') == 'separate' or d.get('cycleway') == 'separate' \
        or d.get('cycleway:right') == 'separate' or d.get('cycleway:left') == 'separate' \
        or d.get('cycleway:both') == 'separate':
            non_cyc2.append((u, v, k))
        if d.get('bicycle') in ['designated', 'use_sidepath']:
            continue
        elif d.get('highway') in ['cycleway'] :
            continue    
        elif d.get('cycleway') in ['track']:
            continue
        elif d.get('cycleway:right') in ['track']:
            continue
        elif d.get('cycleway:left') in ['track']:
            continue
        elif d.get('cycleway:both') in ['track']:
            continue
        non_cyc2.append((u, v, k))
    G6.remove_edges_from(non_cyc2)
    G6 = ox.utils_graph.remove_isolated_nodes(G6)
    #fig, ax = ox.plot_graph(G6)
    stats2 = ox.stats.basic_stats(G6)
    streetlength = stats2['street_length_total']

    scale_factor = 49.74
    scaled_ranges = {
        1: (round(0*scale_factor),round(0*scale_factor)),
        2: (round(1 * scale_factor), round(250 * scale_factor)),  # 1-12435 m (hochskaliert)
        3: (round(251 * scale_factor), round(450 * scale_factor)),# 12486-22383 m (hochskaliert)
        4: (round(451 * scale_factor), round(600 * scale_factor)),# 22434-29844 m (hochskaliert)
        5: (round(601 * scale_factor), round(750 * scale_factor)),# 29895-37305 m (hochskaliert)
        6: (round(751 * scale_factor), round(850 * scale_factor)),# 37356-42273 m (hochskaliert)
        7: (round(851 * scale_factor), round(1100 * scale_factor)),# 42324-54742 m (hochskaliert)
        8: (round(1101 * scale_factor), round(1400 * scale_factor)),# 54793-69628 m (hochskaliert)
        9: (round(1401 * scale_factor), round(1800 * scale_factor)),# 69679-89508 m (hochskaliert)
        10: (round(1801 * scale_factor), round(10000 * scale_factor))# 89559-298440 m (hochskaliert)
    }

    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich

    for key, (low, high) in scaled_ranges.items():
        if low <= streetlength <= high:
            fahrradroute_score = key
            break
    print("Länge der Fahrradroute:", streetlength)
    print("Fahrradroute_Score:", fahrradroute_score)


    ##Connectivity


    inters = ox.stats.intersection_count(G6, min_streets = 2)

    scale_factor = 49.74
    scaled_ranges = {
    1: (round(0 * scale_factor), round(0 * scale_factor)),            # 0 Kreuzungen (hochskaliert)
    2: (round(1), round(1 * scale_factor)),                           # 1-49 Kreuzungen (hochskaliert)
    3: (round(1 * scale_factor + 1), round(3 * scale_factor)),       # 50-149 Kreuzungen (hochskaliert)
    4: (round(3 * scale_factor + 1), round(6 * scale_factor)),       # 150-298 Kreuzungen (hochskaliert)
    5: (round(6 * scale_factor + 1), round(10 * scale_factor)),      # 299-497 Kreuzungen (hochskaliert)
    6: (round(10 * scale_factor + 1), round(15 * scale_factor)),     # 498-746 Kreuzungen (hochskaliert)
    7: (round(15 * scale_factor + 1), round(20 * scale_factor)),     # 747-994 Kreuzungen (hochskaliert)
    8: (round(20 * scale_factor + 1), round(25 * scale_factor)),     # 995-1243 Kreuzungen (hochskaliert)
    9: (round(25 * scale_factor + 1), round(30 * scale_factor)),     # 1244-1492 Kreuzungen (hochskaliert)
    10: (round(30 * scale_factor + 1), round(100 * scale_factor))    # 1493-4974 Kreuzungen (hochskaliert)

    }
    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich
    for key, (low, high) in scaled_ranges.items():
        if low <= inters <= high:
            connectivity_score = key
            break

    print("Anzahl der Kreuzungen:", inters)
    print("Connectivity_Score:", connectivity_score)


    ##Main_Roads


    G9 = ox.graph_from_bbox(north, south, east, west, network_type='drive', simplify=True, retain_all=True, truncate_by_edge=True)
    
    service_edges = [(u, v, k) for u, v, k, d in G9.edges(keys=True, data=True) if d.get('highway') == 'service']

    
    # Liste der zu entfernenden Kanten
    edges_to_remove = []
    for u, v, key, data in G9.edges(keys=True, data=True):
        if data.get('bicycle') == 'separate' or \
        data.get('cycleway') == 'separate' or \
        data.get('cycleway:right') == 'separate' or \
        data.get('cycleway:left') == 'separate' or \
        data.get('cycleway:both') == 'separate' or \
        data.get('bicycle') in ['designated', 'use_sidepath'] or \
        data.get('highway') in ['path','residential', 'cycleway', 'pedestrian', 'track', 'raceway', 'living_street']  or \
        data.get('cycleway') in ['track', 'lane', 'shared_lane'] or \
        data.get('cycleway:right') in ['track', 'lane', 'shared_lane'] or \
        data.get('cycleway:left') in ['track', 'lane', 'shared_lane'] or \
        data.get('cycleway:both') in ['track', 'lane', 'shared_lane']:
            edges_to_remove.append((u, v, key))

    # Entfernen der Kanten aus dem Graphen
    G9.remove_edges_from(edges_to_remove + service_edges)

    # Berechnen der Gesamtlänge der Hauptstraßen
    stats2 = ox.stats.basic_stats(G9)
    length = stats2['street_length_total']


    # Konvertieren des Graphen zu GeoDataFrame
    edges7 = ox.graph_to_gdfs(G9, nodes=False)
    scale_factor = 127.324
    scaled_ranges = {
        1: (round(1101 * scale_factor +1), round(3565 * scale_factor)), 
        2: (round(883 * scale_factor +1), round(1101 * scale_factor)),                                # 1-31957 m (hochskaliert)
        3: (round(726 * scale_factor + 1), round(883 * scale_factor)),           # 31958-54753 m (hochskaliert)
        4: (round(585 * scale_factor + 1), round(726 * scale_factor)),           # 54754-69371 m (hochskaliert)
        5: (round(491 * scale_factor + 1), round(585 * scale_factor)),           # 69372-89356 m (hochskaliert)
        6: (round(405 * scale_factor + 1), round(491 * scale_factor)),           # 89357-111314 m (hochskaliert)
        7: (round(288 * scale_factor + 1), round(405 * scale_factor)),          # 111315-135355 m (hochskaliert)
        8: (round(160 * scale_factor + 1), round(288 * scale_factor)),         # 135356-167219 m (hochskaliert)
        9: (round(1), round(160 * scale_factor)),         # 167220-209380 m (hochskaliert)
        10: (round(0), round(0))         
    }

    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich

    for key, (low, high) in scaled_ranges.items():
        if low <= length <= high:
            main_roads_score = key
            break
    print(f"Main Roads Length: {length}")
    print(f"Main Roads Length Score: {main_roads_score}")


    ## Neue Road Class/Separation Gewichtung

    road_scores2 = {
        'trunk': 0, 'trunk_link': 0,
        'primary': 1, 'primary_link': 1,
        'secondary': 2,'secondary_link': 2,
        'tertiary': 3, 'tertiary_link': 3,
        'secondary': 4,'secondary_link': 4,
        'residential': 7, 'living_street': 7,
        'path': 8, 'track': 8, 'pedestrian': 8,
        'cycleway': 10,
    }


    def calculate_adjusted_road_score(row):
        # Prüfen, ob der 'highway'-Wert eine Liste ist, und den ersten Wert verwenden, falls ja
        highway_value = row['highway'][0] if isinstance(row['highway'], list) else row['highway']
        
        # Basis-Score basierend auf der Straßenklassifizierung
        base_score = road_scores2.get(highway_value, 0)
        
        # Anpassung der Punktezahl basierend auf Fahrradwegbedingungen
        multiplier = 1  # Standardmultiplikator ist 1
        if highway_value == 'cycleway' or \
            (row.get('cycleway') in ['track', 'opposite_track']) or \
            (row.get('cycleway:right') in ['track', 'opposite_track']) or \
            (row.get('cycleway:left') in ['track', 'opposite_track']):
            multiplier = 3
        elif row.get('bicycle_road') == 'yes' or \
            row.get('cycleway:both') == 'lane' or \
            row.get('cycleway') in ['lane', 'opposite_lane'] or \
            row.get('cycleway:right') == 'lane' or \
            row.get('cycleway:left') == 'lane':
            multiplier = 2
        
        # Angepassten Score berechnen
        adjusted_score = base_score * multiplier
        return adjusted_score

    # Berechnen des angepassten Scores für jede Kante
    edges['adjusted_road_separation_score'] = edges.apply(calculate_adjusted_road_score, axis=1)

    # Berechnen der gewichteten Scores
    edges['weighted_road_separation_score'] = edges['adjusted_road_separation_score'] * edges['length']

    # Berechnen des Gesamtwerts der gewichteten Scores und der Gesamtlänge
    total_weighted_road_separation_score = edges['weighted_road_separation_score'].sum()
    total_length = edges['length'].sum()

    # Durchschnittlichen gewichteten Score berechnen
    road_class_separation_score = total_weighted_road_separation_score / total_length

    print(f'Gewichteter Road_Class_Score mit Anpassungen für Radwege: {road_class_separation_score}')


    ##Anteil Grünflächen


   # Funktion zur Bewertung des Grünflächenanteils
    def bewerte_gruenflaechen(anteil):
        if anteil == 0:
            return 0
        elif 0 < anteil <= 9:
            return 2
        elif 9 < anteil <= 22:
            return 4
        elif 22 < anteil <= 41:
            return 6
        elif 41 < anteil <= 61:
            return 8
        elif 61 < anteil:
            return 10
        else:
            return "Anteil außerhalb des definierten Bereichs"

    tags = {'leisure': 'park', 'landuse': ['forest', 'recreation_ground']}

    # Extrahieren von Grünflächen innerhalb der Bounding Box mit OSMnx
    green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)

    # Erstellen eines GeoDataFrame für die Grünflächen
    green_spaces_gdf = gpd.GeoDataFrame(green_spaces, geometry='geometry', crs="EPSG:4326")

    # Transformieren in die flächentreue Projektion EPSG:3035 für die Flächenberechnung
    green_spaces_gdf = green_spaces_gdf.to_crs("EPSG:3035")

    # Berechnen der Gesamtfläche der Grünflächen in Quadratkilometern
    total_green_area = green_spaces_gdf.area.sum() / 10**6

    # Erstellen eines Polygons für die Bounding Box und Berechnen der Gesamtfläche
    bbox_polygon = box(west, south, east, north)
    bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox_polygon]}, crs="EPSG:4326").to_crs("EPSG:3035")
    bbox_area_km2 = bbox_gdf.area.sum() / 10**6

    # Berechnen des prozentualen Anteils der Grünflächen
    percent_green_space = (total_green_area/ bbox_area_km2) * 100


    # Anwenden des Bewertungssystems
    green_score = bewerte_gruenflaechen(percent_green_space)

    print(f"Grünfläche in km²: {total_green_area:.2f}")
    print(f"Anteil der Grünfläche: {percent_green_space:.2f}%")
    print(f"Punkte für Grünflächenanteil: {green_score}")


    
    ###Endbewertung

    Note = (main_roads_score + connectivity_score + fahrradroute_score + slope_score + surfaces_score + separation_score + road_class_score + green_score)/8

    Note2 = (0.1*main_roads_score + 0.23*connectivity_score + 0.17*fahrradroute_score + 0.1*slope_score + 0.05*surfaces_score + 0.19*road_class_separation_score+ 0.16*green_score)
    
    print(f"Endbewertung: {Note}")
    print(f"Endbewertung2: {Note2}")

    results.at[index, 'latitude'] = latitude
    results.at[index, 'longitude'] = longitude
    results.at[index, 'ort'] = ort

    results.at[index, 'Surface_Bewertung'] = surfaces_score
    results.at[index, 'Road_Class_Bewertung'] = road_class_score
    results.at[index, 'Route_Separation_Bewertung'] = separation_score
    results.at[index, 'Steigung_Bewertung'] = slope_score

    results.at[index, 'Main_Roads_Bewertung'] = main_roads_score
    results.at[index, 'Fahrradroute_Length_Bewertung'] = fahrradroute_score
    results.at[index, 'Connectivity_Bewertung'] = connectivity_score

    results.at[index, 'Road_Separation_Bewertung'] = road_class_separation_score
    results.at[index, 'Green_Bewertung'] = green_score


    results.at[index, 'Endbewertung'] = Note
    results.at[index, 'Endbewertung2'] = Note2


results.to_csv('Bewertungsergebnisse_Drost.csv', index=False)

Gesamtbewertungsmethode 2

In [16]:
#Benötigte Module importieren

import osmnx as ox
import math
import pandas as pd
import numpy as np
from osgeo import gdal
import geopandas as gpd
from shapely.geometry import box

# OSMnx konfigurieren und mögliche Tags auswählen

ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

#Datei der Koordinaten in den Code laden

file_path = 'FinalData_processed_Test.csv'
df = pd.read_csv(file_path)

# Ergebnis DataFrame vorbereiten

results = pd.DataFrame()

#Schleife vorbereiten

for index, row in df.iterrows():
   # ort = row['ort']
    latitude, longitude =row['lat'], row['lon']
    
    #Koordinaten einem 5km*5km Gitternetz einteilen
    #Geschätzte anzahl an Metern pro 1° Latitude
    meters_per_lat = 111320
    meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
    delta_lat = 5000/ meters_per_lat
    delta_lon = 5000/ meters_per_lon
    north = latitude + delta_lat
    south = latitude
    east = longitude + delta_lon
    west = longitude
    bbox = (west, south, east, north)
    
    #OSMnx Abfrage für jede Koordinaten, innerhalb der Bounding Box. Network_Type = Bike, um jedes Netzwerk dass mitm Fahrrad erreichbar ist. 
    #Simplify= True, um die Abfrage einfach zu halten und nur Segmente zwischen Kreuzungen betrachtet werden
    #Retain_all = True, um alle Nodes innerhalb der Bounding Box zu berücksichtigen, auch die die nicht mit dem Rest verbunden sind
    #Truncate_by_edge = True, um die Nodes die das nächste Verbindungsstück außerhalb der Bounding Box auch zu berücksichtigen. 

    G1 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    # Entfernen von "service" Edges
    service_edges = [(u, v, k) for u, v, k, d in G1.edges(keys=True, data=True) if d.get('highway') == 'service']

    G1.remove_edges_from(service_edges)

    #Steigung OSMnx Funktion definieren, um Steigungprozente an die Segmente zu ermitteln

    def add_elevation_and_slope(G1, raster_path):
        G1 = ox.elevation.add_node_elevations_raster(G1, raster_path)
        G1 = ox.elevation.add_edge_grades(G1, add_absolute=True)
        return G1
    #Rasterdatei mit Höhenlinien werden eingespielt
    raster_path = "srtm_germany_dtm.tif"
    #Funktion wird auf diese Rasterdatei angewandt
    G1 = add_elevation_and_slope(G1, raster_path)

    ##Steigung Bewertung

    #Umwandlung in GeoDataFrame
    edges = ox.graph_to_gdfs(G1, nodes=False)
    #Negative Grade werden als 0 gewertet, bevor die Steigung sich aufhebt
    edges['grade'] = edges['grade'].apply(lambda x: max(x, 0))
    #Umwandlung in Prozent 
    edges['grade'] = edges['grade'] * 100
    #Funktion zur Bewertung der Steigung (grade)
    def grade_score(grade):
        if grade > 20:
            return 0
        elif 10 < grade <= 20:
            return 1
        elif 7 < grade <= 10:
            return 2
        elif 5 < grade <= 7:
            return 3
        elif 3 < grade <= 5:
            return 4
        elif 2 < grade <= 3:
            return 5
        elif 1 < grade <= 2:
            return 6
        elif 0.5 < grade <= 1:
            return 7
        elif 0.2 < grade <= 0.5:
            return 8
        elif 0 < grade <= 0.2:
            return 9
        else:  # grade == 0
            return 10
        
    # Anwenden der Bewertungsfunktionen auf die Daten  
    edges['grade_score'] = edges['grade'].apply(grade_score)
    # Berechnen der gewichteten Scores (Steigung)
    edges['weighted_grade_score'] = edges['grade_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_grade_score = edges['weighted_grade_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    slope_score = total_weighted_grade_score / total_length

    print(f'Gewichteter Slope_Score: {slope_score}')



    ## Bewertung der Oberfläche (surface)
        
    surface_scores = {
        'asphalt': 10,'concrete': 8,
        'concrete:lanes': 6,'concrete:plates': 6, 'plates': 6, 'paving_stones': 6,
        'paved': 6,
        'compacted': 4, 'dirt': 4, 'unpaved': 4, 'ground' : 4, 'sett': 4, 'metal': 4, 'wood': 4, 'gravel': 4, 'grass': 4,
        'unhewn_cobblestone': 0, 'cobblestone': 0, 'sand': 0,
    }

    # Funktion zur Bewertung der Oberfläche
    def surface_score(surface):
        # Prüfen, ob der Wert eine Liste oder ein Array ist
        if isinstance(surface, list) or isinstance(surface, np.ndarray):
            # Nehmen Sie den ersten Wert der Liste/Array oder geben Sie 0 zurück, wenn die Liste leer ist
            surface_value = surface[0] if surface else None
        else:
            surface_value = surface

        # Standardwert auf 0 setzen, falls die Oberfläche nicht bekannt ist oder fehlt
        return surface_scores.get(surface_value, None)
    
    #Anwendung der Funktion
    edges['surface_score'] = edges['surface'].apply(surface_score)

    edges = edges.dropna(subset = ['surface_score'])
    # Berechnen der gewichteten Scores (Surface)
    edges['weighted_surface_score'] = edges['surface_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_surface_score = edges['weighted_surface_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    surfaces_score = total_weighted_surface_score / total_length

    print(f'Gewichteter Surfaces_Score: {surfaces_score}')



    ##Grünfläche

    
    # Funktion zur Bewertung des Grünflächenanteils
    def bewerte_gruenflaechen(anteil):
        if anteil < 9:
            return 0
        elif 9 < anteil <= 22:
            return 2
        elif 22 < anteil <= 41:
            return 4
        elif 41 < anteil <= 61:
            return 6
        elif 61 < anteil <= 81:
            return 8
        elif 81 < anteil:
            return 10
        else:
            return "Anteil außerhalb des definierten Bereichs"
    
    #Tags um Grünanalagen und Wasserbereiche zu bewerten
    tags = {'natural': ['water','bay', 'coastline','beach'],'leisure': 'park', 'landuse': ['meadow','grass','farmland','forest', 'recreation_ground']}
    tags = {'leisure': 'park', 'landuse': ['grass','forest'], 'natural': ['water','scrub', 'wetland', 'coastline', 'sand']}

    # Extrahieren von Grünflächen innerhalb der Bounding Box mit OSMnx
    green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)

    # Erstellen eines GeoDataFrame für die Grünflächen
    green_spaces_gdf = gpd.GeoDataFrame(green_spaces, geometry='geometry', crs="EPSG:4326")

    # Transformieren in die flächentreue Projektion EPSG:3035 für die Flächenberechnung
    green_spaces_gdf = green_spaces_gdf.to_crs("EPSG:3035")

    # Berechnen der Gesamtfläche der Grünflächen in Quadratkilometern
    total_green_area = green_spaces_gdf.area.sum() / 10**6

    # Erstellen eines Polygons für die Bounding Box und Berechnen der Gesamtfläche
    bbox_polygon = box(west, south, east, north)
    bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox_polygon]}, crs="EPSG:4326").to_crs("EPSG:3035")
    bbox_area_km2 = bbox_gdf.area.sum() / 10**6

    # Berechnen des prozentualen Anteils der Grünflächen
    percent_green_space = (total_green_area/ bbox_area_km2) * 100


    # Anwenden des Bewertungssystems
    green_score = bewerte_gruenflaechen(percent_green_space)

    print(f"Grünfläche in km²: {total_green_area:.2f}")
    print(f"Anteil der Grünfläche: {percent_green_space:.2f}%")
    print(f"Punkte für Grünflächenanteil: {green_score}")


    ##Connectivity

    G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    #Zu Beachtenden und enfernten Tags
    non_cyc2 = []
    for u, v, k, d in G6.edges(keys=True, data=True):

        #Tags mit separate werden entfernt um Dopplung zu vermeiden, da sonst Straße und extra Kodierung der Nebenstraße 
        if d.get('bicycle') == 'separate' or d.get('cycleway') == 'separate' \
        or d.get('cycleway:right') == 'separate' or d.get('cycleway:left') == 'separate' \
        or d.get('cycleway:both') == 'separate':
            non_cyc2.append((u, v, k))
        if d.get('bicycle') in ['designated', 'use_sidepath']:
            continue
        elif d.get('highway') in ['cycleway'] :
            continue    
        elif d.get('cycleway') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:right') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:left') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:both') in ['track', 'opposite_track']:
            continue   
        elif d.get('highway') == 'path' and d.get('bicycle') == 'yes':
            continue
        non_cyc2.append((u, v, k))
    G6.remove_edges_from(non_cyc2)
    G6 = ox.utils_graph.remove_isolated_nodes(G6)


    #Funktion zum zählen der Kreuzungen, die mindestens 2 Straßen beinhaltet
    inters = ox.stats.intersection_count(G6, min_streets = 2)

    #Der Berechnete Scale-Faktor, der für repräsentative Anwendung errechnet wurde
    scale_factor = 49.74
    #Einteilung
    scaled_ranges = {
    0: (round(0 * scale_factor), round(0 * scale_factor)),            # 0 Kreuzungen (hochskaliert)
    1: (round(1), round(1 * scale_factor)),                           # 1-49 Kreuzungen (hochskaliert)
    2: (round(1 * scale_factor + 1), round(3 * scale_factor)),       # 50-149 Kreuzungen (hochskaliert)
    3: (round(3 * scale_factor + 1), round(6 * scale_factor)),       # 150-298 Kreuzungen (hochskaliert)
    4: (round(6 * scale_factor + 1), round(10 * scale_factor)),      # 299-497 Kreuzungen (hochskaliert)
    5: (round(10 * scale_factor + 1), round(15 * scale_factor)),     # 498-746 Kreuzungen (hochskaliert)
    6: (round(15 * scale_factor + 1), round(20 * scale_factor)),     # 747-994 Kreuzungen (hochskaliert)
    7: (round(20 * scale_factor + 1), round(25 * scale_factor)),     # 995-1243 Kreuzungen (hochskaliert)
    8: (round(25 * scale_factor + 1), round(30 * scale_factor)),     # 1244-1492 Kreuzungen (hochskaliert)
    9: (round(30 * scale_factor + 1), round(55 * scale_factor)),    # 1493- 2723 Kreuzungen (hochskaliert)
    10: (round(55 * scale_factor + 1), round(100 * scale_factor)) #2723 - 
    }
    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich
    for key, (low, high) in scaled_ranges.items():
        if low <= inters <= high:
            connectivity_score = key
            break

    print("Anzahl der Kreuzungen:", inters)
    print("Connectivity_Score:", connectivity_score)



    ##Hauptstraßeninfrastruktur in Relation
    try: 
    #Abfrage mit Custom_Filter, der bestimmte highway-tags abfrägt
        graph = ox.graph_from_bbox(north, south, east, west, network_type='bike', custom_filter='["highway"~"primary|primary_link|secondary|secondary_link"]', simplify=False, retain_all=True, truncate_by_edge=True)

        # Konvertiere das Graphenobjekt in ein GeoDataFrame
        edges = ox.graph_to_gdfs(graph, nodes=False)
        if edges.empty:
        # Keine Hauptstraßen vorhanden, überspringe diese Kategorie in der Endbewertung
            hauptstraße_score = None
        else:
        # Zuerst wird eine Funktion erstellt, die prüft, ob die Spalte existiert und den Filter anwendet
            def check_and_filter(df, column, value):
                if column in df.columns:
                    return df[column] == value
                else:
                    return pd.Series([False] * len(df), index=df.index)

            # Dann wird die Filterlogik angepasst, um diese Funktion zu nutzen
            bike_infrastructure_filter = (
                check_and_filter(edges, 'cycleway', 'lane') |
                check_and_filter(edges, 'bicycle', 'designated') |
                check_and_filter(edges, 'cycleway', 'track') |
                check_and_filter(edges, 'cycleway:right', 'track') |
                check_and_filter(edges, 'cycleway:left', 'track') |
                check_and_filter(edges, 'cycleway:both', 'track') |
                check_and_filter(edges, 'cycleway:right', 'lane') |
                check_and_filter(edges, 'cycleway:left', 'lane') |
                check_and_filter(edges, 'cycleway:both', 'lane') |
                check_and_filter(edges, 'bicycle', 'use_sidepath') |
                check_and_filter(edges, 'cycleway', 'separate') |
                check_and_filter(edges, 'cycleway:right', 'separate') |
                check_and_filter(edges, 'cycleway:left', 'separate') |
                check_and_filter(edges, 'cycleway:both', 'separate') 
            )

            bike_lanes = edges[bike_infrastructure_filter]

            # Berechnung der Fahrradspurenlänge
            # Summe der Gesamtlänge und die angepasste Länge
            total_length = bike_lanes['length'].sum()

            # Berechnung der Gesamtlänge aller primären und sekundären Straßen
            all_edges_length = edges['length'].sum()

            # Ausgabe der Ergebnisse
            print(f"Gesamtlänge der Fahrradinfrastruktur: {total_length} Meter")
            print(f"Gesamtlänge aller primären und sekundären Straßen: {all_edges_length} Meter")

            # Berechnung des Verhältnisses
            ratio = total_length / all_edges_length * 100 if all_edges_length > 0 else 0
            print(f"Verhältnis: {ratio}%")
            
            #Funktion für Bewertung des Indikators
            def bewerte_hauptstraße(anteil):
                if anteil == 0:
                    return 0
                elif 0 < anteil <= 21:
                    return 2
                elif 21 < anteil <= 47:
                    return 4
                elif 47 < anteil <= 67:
                    return 6
                elif 67 < anteil <= 86:
                    return 8
                elif 86 < anteil:
                    return 10
                else:
                    return "Anteil außerhalb des definierten Bereichs"
        
        # Ermittlung der Bewertung
        # Standardwert für Werte über dem höchsten Bereich

            hauptstraße_score = bewerte_hauptstraße(ratio)
            print("Bewertungspunkt:", hauptstraße_score)

    except Exception as e:
    # Behandlung für den Fall, dass keine Daten zurückgegeben werden oder ein anderer Fehler auftritt
        print(f"Ein Fehler ist aufgetreten: {e}")
        hauptstraße_score = None

    ##Fahrradfreundliche Straßen Verhältniss

    G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    #Service Edges werden wieder entfernt
    service_edges = [(u, v, k) for u, v, k, d in G6.edges(keys=True, data=True) if d.get('highway') == 'service']
    G6.remove_edges_from(service_edges)
    G6 = ox.utils_graph.remove_isolated_nodes(G6)

    #Umwandlung in GeoDatenFrame
    alledges = ox.graph_to_gdfs(G6, nodes=False)
    #Länge alle Straßen/außer Service
    total_length_all = alledges['length'].sum()

    #Abfrage
    G7 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    #Fahrradfreundliche Straßen filtern
    non_cyc2 =[]
    for u, v, k, d in G7.edges(keys=True, data=True):
        if d.get('bicycle') in ['designated','use_sidepath']:
            continue
        elif d.get('highway') in ['cycleway', 'residential', 'tertiary','tertiary_link', 'living_street','unclassified'] :
            continue 
        elif d.get('highway') == 'track' and d.get('tracktype') in ['grade1','grade2','grade3']:
            continue
        elif d.get('cycleway') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:right') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:left') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:both') in ['track', 'opposite_track']:
            continue   
        elif d.get('highway') == 'path' and d.get('bicycle') == 'yes':
            continue
        elif d.get('bicycle_road') == 'yes':
            continue
        non_cyc2.append((u, v, k))

                #getrennter Fußweg und Radweg auf einem Weg highway=path + bicycle=designated + foot=designated + segregated=yes
                #gemeinsamer Fußweg und Radweg auf einem Weg highway=path + bicycle=designated + foot=designated + segregated=no
    
    #Entfernung aller Straßen die nicht den Bedingung haben
    G7.remove_edges_from(non_cyc2 + service_edges)
    G7 = ox.utils_graph.remove_isolated_nodes(G7)

    #Umwandlung in GeoDatenFrame
    bikefriendlyedges = ox.graph_to_gdfs(G7, nodes=False)

    #Länge der fahrradfreundlichen Straßen
    total_length = bikefriendlyedges['length'].sum()
    # Verwenden Sie eine sicherere Methode, um die angepasste Länge zu berechnen
    def calculate_adjusted_length(row):
        # Prüfen, ob die notwendigen Spalten existieren und die Bedingungen erfüllen
        conditions = [
            row.get('highway') == 'cycleway',
        ]
        # Verdoppeln Sie die Länge, wenn eine der Bedingungen erfüllt ist
        if any(conditions):
            return row['length'] * 2
        else:
            return row['length']

    # Berechnung der angepasste Länge für jede Zeile
    bikefriendlyedges['adjusted_length'] = bikefriendlyedges.apply(calculate_adjusted_length, axis=1)

    # komplette Länge mit angepasster Länge
    adjusted_total_length = bikefriendlyedges['adjusted_length'].sum()

    #Prozentuales Verhältnis für Fahrradfreundliche Straßen
    bike_friendly_street_share = adjusted_total_length/total_length_all * 100

    #Funktion für die Bewertung der Fahrradfreundlichkeit
    def bewerte_fahrradfreund(anteil):
        if anteil == 0:
            return 0
        elif 0 < anteil <= 26:
            return 2
        elif 26 < anteil <= 58:
            return 4
        elif 58 < anteil <= 75:
            return 6
        elif 75 < anteil <= 87:
            return 8
        elif 87 < anteil:
            return 10
        else:
            return "Anteil außerhalb des definierten Bereichs"
    
    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich
    fahrradroute_score = bewerte_fahrradfreund(bike_friendly_street_share)

    print("Länge der Fahrradroute ohne Gewichtung:", total_length )
    print("Länge der Fahrradroute:", adjusted_total_length)
    print("Länge Insgesamt:", total_length_all)
    print("Prozentualer Share:", bike_friendly_street_share)
    print("Bewertungspunkt:", fahrradroute_score)



    ##Endbewertung für jedes Gitternetz, Aber mit Abfrage ob es einen Hauptstraßenscore gibt
    if hauptstraße_score is not None:
    # Berechnung der Endbewertung mit Hauptstraßenbewertung
        Note = (0.15*connectivity_score + 0.20*fahrradroute_score + 0.16*slope_score + 0.13*green_score + 0.25*hauptstraße_score + 0.11*surfaces_score) / (0.15 + 0.20 + 0.16 + 0.13 + 0.25 + 0.11)
    else:
    # Berechnung der Endbewertung ohne Hauptstraßenbewertung, Anpassung der Gewichtungen notwendig
        total_weight = 0.15 + 0.20 + 0.16 + 0.13 + 0.11  # Ohne Hauptstraße
        Note = (0.15*connectivity_score + 0.20*fahrradroute_score + 0.16*slope_score + 0.13*green_score + 0.11*surfaces_score) / total_weight
    
    
    print(f"Endbewertung: {Note}")

    #Festsetzung der neuen Spalten und Übertragung der Inhalte
    df.at[index, 'Hauptstraße_Bewertung'] = hauptstraße_score
    df.at[index, 'Steigung_Bewertung'] = slope_score
    df.at[index, 'Surface_Bewertung'] = surfaces_score
    df.at[index, 'Fahrradroute_Relation_Bewertung'] = fahrradroute_score
    df.at[index, 'Connectivity_Bewertung'] = connectivity_score
    df.at[index, 'Green_Bewertung'] = green_score
    df.at[index, 'Endbewertung'] = Note

# Speichern des aktualisierten DataFrames in derselben CSV-Datei, um sie zu überschreiben
df.to_csv('FinalData_processed_Test.csv', index=False)

C:\Users\kevdr\AppData\Local\Temp\ipykernel_24952\3884088359.py:13: UserWarning: The `utils.config` function is deprecated and will be removed in a future release. Instead, use the `settings` module directly to configure a global setting's value. For example, `ox.settings.log_console=True`.
  ox.config(use_cache=True, log_console=True)


Gewichteter Slope_Score: 7.270289954803759
Gewichteter Surfaces_Score: 7.384546662739459


C:\Users\kevdr\AppData\Local\Temp\ipykernel_24952\3884088359.py:183: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in km²: 34.57
Anteil der Grünfläche: 138.27%
Punkte für Grünflächenanteil: 10
Anzahl der Kreuzungen: 19
Connectivity_Score: 1
Gesamtlänge der Fahrradinfrastruktur: 0.0 Meter
Gesamtlänge aller primären und sekundären Straßen: 2431.344 Meter
Verhältnis: 0.0%
Bewertungspunkt: 0
Länge der Fahrradroute ohne Gewichtung: 218379.071
Länge der Fahrradroute: 218379.071
Länge Insgesamt: 299289.713
Prozentualer Share: 72.96577914791212
Bewertungspunkt: 6
Endbewertung: 4.625546525669941
Gewichteter Slope_Score: 7.143788946528515
Gewichteter Surfaces_Score: 7.408272156838516


C:\Users\kevdr\AppData\Local\Temp\ipykernel_24952\3884088359.py:183: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in km²: 33.35
Anteil der Grünfläche: 133.39%
Punkte für Grünflächenanteil: 10
Anzahl der Kreuzungen: 73
Connectivity_Score: 2
Gesamtlänge der Fahrradinfrastruktur: 2032.2340000000002 Meter
Gesamtlänge aller primären und sekundären Straßen: 23601.108 Meter
Verhältnis: 8.610756749217028%
Bewertungspunkt: 2
Länge der Fahrradroute ohne Gewichtung: 243955.285
Länge der Fahrradroute: 243955.285
Länge Insgesamt: 325752.557
Prozentualer Share: 74.88975289916144
Bewertungspunkt: 6
Endbewertung: 5.2579161686967995
Gewichteter Slope_Score: 6.86229608132755
Gewichteter Surfaces_Score: 6.581546925226751


C:\Users\kevdr\AppData\Local\Temp\ipykernel_24952\3884088359.py:183: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in km²: 65.66
Anteil der Grünfläche: 262.62%
Punkte für Grünflächenanteil: 10
Anzahl der Kreuzungen: 18
Connectivity_Score: 1
Gesamtlänge der Fahrradinfrastruktur: 0.0 Meter
Gesamtlänge aller primären und sekundären Straßen: 14314.115000000002 Meter
Verhältnis: 0.0%
Bewertungspunkt: 0
Länge der Fahrradroute ohne Gewichtung: 155266.641
Länge der Fahrradroute: 156019.685
Länge Insgesamt: 301484.652
Prozentualer Share: 51.75045693536665
Bewertungspunkt: 4
Endbewertung: 4.0719375347873505


Gesamtbewertungsmethode 3(mit Vereinfachungen) FINALE VERSION

In [1]:
#Benötigte Module importieren

import osmnx as ox
import math
import pandas as pd
import numpy as np
from osgeo import gdal
import geopandas as gpd
from shapely.geometry import box
import copy

# OSMnx konfigurieren und mögliche Tags auswählen

ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse','crossing',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

#Datei der Koordinaten in den Code laden

file_path = 'NeueGewichtung_Min20Data_Deutschland.csv'
df = pd.read_csv(file_path)

# Ergebnis DataFrame vorbereiten

results = pd.DataFrame()

# Ergänzung der neuen Spalten, wenn sie noch nicht existieren
for column in ['Hauptstraße_Bewertung', 'Steigung_Bewertung', 'Surface_Bewertung', 'Fahrradroute_Relation_Bewertung', 'Connectivity_Bewertung', 'Green_Bewertung', 'Endbewertung']:
    if column not in df.columns:
        df[column] = np.nan

#Schleife vorbereiten
start_zeile = 2650


for index, row in df.iloc[start_zeile:].iterrows():
   # ort = row['ort']
    latitude, longitude =row['lat'], row['lon']
    
    #Koordinaten einem 5km*5km Gitternetz einteilen
    #Geschätzte anzahl an Metern pro 1° Latitude
    meters_per_lat = 111320
    meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
    delta_lat = 5000/ meters_per_lat
    delta_lon = 5000/ meters_per_lon
    north = latitude + delta_lat
    south = latitude
    east = longitude + delta_lon
    west = longitude
    bbox = (west, south, east, north)
    print(f'latitude: {latitude}',f'longitue: {longitude}')
    #OSMnx Abfrage für jede Koordinaten, innerhalb der Bounding Box. Network_Type = Bike, um jedes Netzwerk dass mitm Fahrrad erreichbar ist. 
    #Simplify= True, um die Abfrage einfach zu halten und nur Segmente zwischen Kreuzungen betrachtet werden
    #Retain_all = True, um alle Nodes innerhalb der Bounding Box zu berücksichtigen, auch die die nicht mit dem Rest verbunden sind
    #Truncate_by_edge = True, um die Nodes die das nächste Verbindungsstück außerhalb der Bounding Box auch zu berücksichtigen. 

    G1 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    # Entfernen von "service" Edges
  
    service_edges = [(u, v, k) for u, v, k, d in G1.edges(keys=True, data=True) if d.get('highway') == 'service']

    G1.remove_edges_from(service_edges)
    G_original = copy.deepcopy(G1) #Kopie um Abfragevolumen zu verrringern

    edges = ox.graph_to_gdfs(G1, nodes=False) # In GeoDataFrame umwandeln
    ## Bewertung der Oberfläche (surface)
        
    surface_scores = {
        'asphalt': 10,'concrete': 8,
        'concrete:lanes': 6,'concrete:plates': 6, 'plates': 6, 'paving_stones': 6,
        'paved': 6,
        'compacted': 4, 'dirt': 4, 'unpaved': 4, 'ground' : 4, 'sett': 4, 'metal': 4, 'wood': 4, 'gravel': 4, 'grass': 4,
        'unhewn_cobblestone': 0, 'cobblestone': 0, 'sand': 0,
    }

    # Funktion zur Bewertung der Oberfläche
    def surface_score(surface):
        # Prüfen, ob der Wert eine Liste oder ein Array ist
        if isinstance(surface, list) or isinstance(surface, np.ndarray):
            # Nehmen Sie den ersten Wert der Liste/Array oder geben Sie 0 zurück, wenn die Liste leer ist
            surface_value = surface[0] if surface else None
        else:
            surface_value = surface

        # Standardwert auf 0 setzen, falls die Oberfläche nicht bekannt ist oder fehlt
        return surface_scores.get(surface_value, None)
    
    #Anwendung der Funktion
    edges['surface_score'] = edges['surface'].apply(surface_score)

    edges = edges.dropna(subset = ['surface_score'])
    # Berechnen der gewichteten Scores (Surface)
    edges['weighted_surface_score'] = edges['surface_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_surface_score = edges['weighted_surface_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    surfaces_score = total_weighted_surface_score / total_length

    print(f'Gewichteter Surfaces_Score: {surfaces_score}')


 #Steigung OSMnx Funktion definieren, um Steigungprozente an die Segmente zu ermitteln

    def add_elevation_and_slope(G1, raster_path):
        G1 = ox.elevation.add_node_elevations_raster(G1, raster_path)
        G1 = ox.elevation.add_edge_grades(G1, add_absolute=True)
        return G1
    #Rasterdatei mit Höhenlinien werden eingespielt
    raster_path = "srtm_germany_dtm.tif"
    #Funktion wird auf diese Rasterdatei angewandt
    G1 = add_elevation_and_slope(G1, raster_path)

    ##Steigung Bewertung

    #Umwandlung in GeoDataFrame
    edges = ox.graph_to_gdfs(G1, nodes=False)
    #Negative Grade werden als 0 gewertet, bevor die Steigung sich aufhebt
    #edges['grade'] = edges['grade'].apply(lambda x: max(x, 0))
    #Negative Steigung wird entfernt und nicht bewertet, da diese die eigenttliche Steigung beeinflussen
    edges = edges[edges['grade'] >= 0]
    #Umwandlung in Prozent 
    edges['grade'] = edges['grade'] * 100
    #Funktion zur Bewertung der Steigung (grade)
    def grade_score(grade):
        if grade > 20:
            return 0
        elif 10 < grade <= 20:
            return 1
        elif 7 < grade <= 10:
            return 2
        elif 5 < grade <= 7:
            return 3
        elif 3 < grade <= 5:
            return 4
        elif 2 < grade <= 3:
            return 5
        elif 1 < grade <= 2:
            return 6
        elif 0.5 < grade <= 1:
            return 7
        elif 0.2 < grade <= 0.5:
            return 8
        elif 0 < grade <= 0.2:
            return 9
        else:  # grade == 0
            return 10
        
    # Anwenden der Bewertungsfunktionen auf die Daten  
    edges['grade_score'] = edges['grade'].apply(grade_score)
    # Berechnen der gewichteten Scores (Steigung)
    edges['weighted_grade_score'] = edges['grade_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_grade_score = edges['weighted_grade_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    slope_score = total_weighted_grade_score / total_length

    print(f'Gewichteter Slope_Score: {slope_score}')



    ##Grünfläche

    
    # Funktion zur Bewertung des Grünflächenanteils
    def bewerte_gruenflaechen(anteil):
        if anteil < 9:
            return 0
        elif 9 < anteil <= 22:
            return 2
        elif 22 < anteil <= 41:
            return 4
        elif 41 < anteil <= 61:
            return 6
        elif 61 < anteil <= 81:
            return 8
        elif 81 < anteil:
            return 10
        else:
            return "Anteil außerhalb des definierten Bereichs"
    #
    #Tags um Grünanalagen und Wasserbereiche zu bewerten
    #tags = {'natural': ['water','bay', 'coastline','beach'],'leisure': 'park', 'landuse': ['meadow','grass','farmland','forest', 'recreation_ground']}
    tags = {'leisure': 'park', 'landuse': ['grass','forest','meadow','recreation_ground'], 'natural': ['water','scrub', 'wetland', 'coastline', 'sand']}

    # Extrahieren von Grünflächen innerhalb der Bounding Box mit OSMnx
    green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)

    # Erstellen eines GeoDataFrame für die Grünflächen
    green_spaces_gdf = gpd.GeoDataFrame(green_spaces, geometry='geometry', crs="EPSG:4326")

    # Transformieren in die flächentreue Projektion EPSG:3035 für die Flächenberechnung
    green_spaces_gdf = green_spaces_gdf.to_crs("EPSG:3035")

    # Berechnen der Gesamtfläche der Grünflächen in Quadratkilometern
    total_green_area = green_spaces_gdf.area.sum() / 10**6

    # Erstellen eines Polygons für die Bounding Box und Berechnen der Gesamtfläche
    bbox_polygon = box(west, south, east, north)
    bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox_polygon]}, crs="EPSG:4326").to_crs("EPSG:3035")
    bbox_area_km2 = bbox_gdf.area.sum() / 10**6

    # Berechnen des prozentualen Anteils der Grünflächen
    percent_green_space = (total_green_area/ bbox_area_km2) * 100


    # Anwenden des Bewertungssystems
    green_score = bewerte_gruenflaechen(percent_green_space)

    print(f"Grünfläche in km²: {total_green_area:.2f}")
    print(f"Anteil der Grünfläche: {percent_green_space:.2f}%")
    print(f"Punkte für Grünflächenanteil: {green_score}")


    ##Connectivity

    #G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    #Zu Beachtenden und enfernten Tags
    non_cyc2 = []
    for u, v, k, d in G1.edges(keys=True, data=True):

        #Tags mit separate werden entfernt um Dopplung zu vermeiden, da sonst Straße und extra Kodierung der Nebenstraße 
        if d.get('bicycle') == 'separate' or d.get('cycleway') == 'separate' \
        or d.get('cycleway:right') == 'separate' or d.get('cycleway:left') == 'separate' \
        or d.get('cycleway:both') == 'separate':
            non_cyc2.append((u, v, k))
        if d.get('bicycle') in ['designated', 'use_sidepath']:
            continue
        elif d.get('highway') in ['cycleway'] :
            continue    
        elif d.get('cycleway') in ['track','lane', 'opposite_track']:
            continue
        elif d.get('cycleway:right') in ['track','lane', 'opposite_track']:
            continue
        elif d.get('cycleway:left') in ['track','lane', 'opposite_track']:
            continue
        elif d.get('cycleway:both') in ['track','lane', 'opposite_track']:
            continue   
        elif d.get('highway') == 'path' and d.get('bicycle') == 'yes':
            continue
        elif d.get('bicycle_road') == 'yes':
            continue
        non_cyc2.append((u, v, k))
    G_original.remove_edges_from(non_cyc2)
    G1 = ox.utils_graph.remove_isolated_nodes(G_original)


    #Funktion zum zählen der Kreuzungen, die mindestens 2 Straßen beinhaltet
    inters = ox.stats.intersection_count(G1, min_streets = 2)

    #Der Berechnete Scale-Faktor, der für repräsentative Anwendung errechnet wurde
    scale_factor = 49.74
    #Einteilung
    scaled_ranges = {
    0: (round(0 * scale_factor), round(0 * scale_factor)),            # 0 Kreuzungen (hochskaliert)
    1: (round(1), round(1 * scale_factor)),                           # 1-49 Kreuzungen (hochskaliert)
    2: (round(1 * scale_factor + 1), round(3 * scale_factor)),       # 50-149 Kreuzungen (hochskaliert)
    3: (round(3 * scale_factor + 1), round(6 * scale_factor)),       # 150-298 Kreuzungen (hochskaliert)
    4: (round(6 * scale_factor + 1), round(10 * scale_factor)),      # 299-497 Kreuzungen (hochskaliert)
    5: (round(10 * scale_factor + 1), round(15 * scale_factor)),     # 498-746 Kreuzungen (hochskaliert)
    6: (round(15 * scale_factor + 1), round(20 * scale_factor)),     # 747-994 Kreuzungen (hochskaliert)
    7: (round(20 * scale_factor + 1), round(25 * scale_factor)),     # 995-1243 Kreuzungen (hochskaliert)
    8: (round(25 * scale_factor + 1), round(30 * scale_factor)),     # 1244-1492 Kreuzungen (hochskaliert)
    9: (round(30 * scale_factor + 1), round(55 * scale_factor)),    # 1493- 2723 Kreuzungen (hochskaliert)
    10: (round(55 * scale_factor + 1), round(100 * scale_factor)) #2723 - 
    }
    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich
    for key, (low, high) in scaled_ranges.items():
        if low <= inters <= high:
            connectivity_score = key
            break

    print("Anzahl der Kreuzungen:", inters)
    print("Connectivity_Score:", connectivity_score)



    ##Hauptstraßeninfrastruktur in Relation
    try: 
    #Abfrage mit Custom_Filter, der bestimmte highway-tags abfrägt
        graph = ox.graph_from_bbox(north, south, east, west, network_type='bike', custom_filter='["highway"~"primary|primary_link|secondary|secondary_link"]', simplify=False, retain_all=True, truncate_by_edge=True)

        # Konvertiere das Graphenobjekt in ein GeoDataFrame
        edges = ox.graph_to_gdfs(graph, nodes=False)
        if edges.empty:#Falls keine Hahptstraße vorhanden(kommt selten vor), wird die Gitterzelle übersprungen
            print("Keine Hauptstraßen vorhanden, diese Gitterzelle wird übersprungen.")
            continue
        else:
        # Zuerst wird eine Funktion erstellt, die prüft, ob die Spalte existiert und den Filter anwendet
            def check_and_filter(df, column, value):
                if column in df.columns:
                    return df[column] == value
                else:
                    return pd.Series([False] * len(df), index=df.index)

            # Dann wird die Filterlogik angepasst, um diese Funktion zu nutzen
            bike_infrastructure_filter = (
                check_and_filter(edges, 'cycleway', 'lane') |
                check_and_filter(edges, 'bicycle', 'designated') |
                check_and_filter(edges, 'cycleway', 'track') |
                check_and_filter(edges, 'cycleway:right', 'track') |
                check_and_filter(edges, 'cycleway:left', 'track') |
                check_and_filter(edges, 'cycleway:both', 'track') |
                check_and_filter(edges, 'cycleway:right', 'lane') |
                check_and_filter(edges, 'cycleway:left', 'lane') |
                check_and_filter(edges, 'cycleway:both', 'lane') |
                check_and_filter(edges, 'bicycle', 'use_sidepath') |
                check_and_filter(edges, 'cycleway', 'separate') |
                check_and_filter(edges, 'cycleway:right', 'separate') |
                check_and_filter(edges, 'cycleway:left', 'separate') |
                check_and_filter(edges, 'cycleway:both', 'separate') 
            )

            bike_lanes = edges[bike_infrastructure_filter]

            # Berechnung der Fahrradspurenlänge
            # Summe der Gesamtlänge und die angepasste Länge
            total_length = bike_lanes['length'].sum()

            # Berechnung der Gesamtlänge aller primären und sekundären Straßen
            all_edges_length = edges['length'].sum()

            # Ausgabe der Ergebnisse
            print(f"Gesamtlänge der Fahrradinfrastruktur: {total_length} Meter")
            print(f"Gesamtlänge aller primären und sekundären Straßen: {all_edges_length} Meter")

            # Berechnung des Verhältnisses
            ratio = total_length / all_edges_length * 100 if all_edges_length > 0 else 0
            print(f"Verhältnis: {ratio}%")
            
            #Funktion für Bewertung des Indikators
            def bewerte_hauptstraße(anteil):
                if anteil == 0:
                    return 0
                elif 0 < anteil <= 21:
                    return 2
                elif 21 < anteil <= 47:
                    return 4
                elif 47 < anteil <= 67:
                    return 6
                elif 67 < anteil <= 86:
                    return 8
                elif 86 < anteil:
                    return 10
                else:
                    return "Anteil außerhalb des definierten Bereichs"
        
        # Ermittlung der Bewertung
        # Standardwert für Werte über dem höchsten Bereich

            hauptstraße_score = bewerte_hauptstraße(ratio)
            print("Bewertungspunkt:", hauptstraße_score)

    except Exception as e:
    # Behandlung für den Fall, dass keine Daten zurückgegeben werden oder ein anderer Fehler auftritt
        print(f"Ein Fehler ist aufgetreten: {e}")
        continue

    ##Fahrradfreundliche Straßen Verhältniss

    G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    #Service Edges werden wieder entfernt
    service_edges = [(u, v, k) for u, v, k, d in G6.edges(keys=True, data=True) if d.get('highway') == 'service']
    G6.remove_edges_from(service_edges)
    G6 = ox.utils_graph.remove_isolated_nodes(G6)

    #Umwandlung in GeoDatenFrame
    alledges = ox.graph_to_gdfs(G6, nodes=False)
    #Länge alle Straßen/außer Service
    total_length_all = alledges['length'].sum()

    #Abfrage
    #G7 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

                #getrennter Fußweg und Radweg auf einem Weg highway=path + bicycle=designated + foot=designated + segregated=yes
                #gemeinsamer Fußweg und Radweg auf einem Weg highway=path + bicycle=designated + foot=designated + segregated=no
    #Fahrradfreundliche Straßen filtern
    non_cyc2 =[]
    for u, v, k, d in G6.edges(keys=True, data=True):
        if d.get('bicycle') in ['designated','use_sidepath']:
            continue
        elif d.get('highway') in ['cycleway', 'residential', 'tertiary','tertiary_link', 'living_street','unclassified'] :
            continue 
        elif d.get('highway') == 'track' and d.get('tracktype') in ['grade1','grade2','grade3']:
            continue #unterscheide die verschiedenen Arten von Wirtschaftswegen('Track') und benotet die abstufend anhand der Befahrbarkeit
        elif d.get('cycleway') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:right') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:left') in ['track', 'opposite_track']:
            continue
        elif d.get('cycleway:both') in ['track', 'opposite_track']:
            continue   
        elif d.get('highway') == 'path' and d.get('bicycle') == 'yes':
            continue
        elif d.get('bicycle_road') == 'yes':
            continue
        non_cyc2.append((u, v, k))

    #Entfernung aller Straßen die nicht den Bedingung haben
    G6.remove_edges_from(non_cyc2 + service_edges)
    G6 = ox.utils_graph.remove_isolated_nodes(G6)

    #Umwandlung in GeoDatenFrame
    bikefriendlyedges = ox.graph_to_gdfs(G6, nodes=False)

    #Länge der fahrradfreundlichen Straßen
    total_length = bikefriendlyedges['length'].sum()
    # Verwenden Sie eine sicherere Methode, um die angepasste Länge zu berechnen
    def calculate_adjusted_length(row):
        # Prüfen, ob die notwendigen Spalten existieren und die Bedingungen erfüllen
        conditions = [
            row.get('highway') == 'cycleway',
        ]
        # Verdoppeln Sie die Länge, wenn eine der Bedingungen erfüllt ist
        if any(conditions):
            return row['length'] * 2
        else:
            return row['length']

    # Berechnung der angepasste Länge für jede Zeile
    bikefriendlyedges['adjusted_length'] = bikefriendlyedges.apply(calculate_adjusted_length, axis=1)

    # komplette Länge mit angepasster Länge
    adjusted_total_length = bikefriendlyedges['adjusted_length'].sum()

    #Prozentuales Verhältnis für Fahrradfreundliche Straßen
    bike_friendly_street_share = adjusted_total_length/total_length_all * 100

    #Funktion für die Bewertung der Fahrradfreundlichkeit
    def bewerte_fahrradfreund(anteil):
        if anteil == 0:
            return 0
        elif 0 < anteil <= 26:
            return 2
        elif 26 < anteil <= 58:
            return 4
        elif 58 < anteil <= 75:
            return 6
        elif 75 < anteil <= 87:
            return 8
        elif 87 < anteil:
            return 10
        else:
            return "Anteil außerhalb des definierten Bereichs"
    
    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich
    fahrradroute_score = bewerte_fahrradfreund(bike_friendly_street_share)

    print("Länge der Fahrradroute ohne Gewichtung:", total_length )
    print("Länge der Fahrradroute:", adjusted_total_length)
    print("Länge Insgesamt:", total_length_all)
    print("Prozentualer Share:", bike_friendly_street_share)
    print("Bewertungspunkt:", fahrradroute_score)



    ##Endbewertung für jedes Gitternetz, Aber mit Abfrage ob es einen Hauptstraßenscore gibt

    # Berechnung der Endbewertung mit Hauptstraßenbewertung
    Note = (0.13*connectivity_score + 0.21*fahrradroute_score + 0.14*slope_score + 0.13*green_score + 0.26*hauptstraße_score + 0.13*surfaces_score)

    print(f"Endbewertung: {Note}")

    #Festsetzung der neuen Spalten und Übertragung der Inhalte
    df.at[index, 'Hauptstraße_Bewertung'] = hauptstraße_score
    df.at[index, 'Steigung_Bewertung'] = slope_score
    df.at[index, 'Surface_Bewertung'] = surfaces_score
    df.at[index, 'Fahrradroute_Relation_Bewertung'] = fahrradroute_score
    df.at[index, 'Connectivity_Bewertung'] = connectivity_score
    df.at[index, 'Green_Bewertung'] = green_score
    df.at[index, 'Endbewertung'] = Note

        # Nach jeder 10. Iteration speichern
    if (index + 1) % 10 == 0:
        df.to_csv('NeueGewichtung_Min20Data_Deutschland.csv', index=False)
        print(f'Daten nach {index + 1} Zeilen gespeichert.')
        
# Speichern des aktualisierten DataFrames in derselben CSV-Datei, um sie zu überschreiben
df.to_csv('NeueGewichtung_Min20Data_Deutschland.csv', index=False)

C:\Users\kevdr\AppData\Local\Temp\ipykernel_20616\1858481351.py:14: UserWarning: The `utils.config` function is deprecated and will be removed in a future release. Instead, use the `settings` module directly to configure a global setting's value. For example, `ox.settings.log_console=True`.
  ox.config(use_cache=True, log_console=True)


latitude: 53.57263606326293 longitue: 10.060373907027014
Gewichteter Surfaces_Score: 9.04577193707507
Gewichteter Slope_Score: 8.096380683443522


C:\Users\kevdr\AppData\Local\Temp\ipykernel_20616\1858481351.py:196: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in km²: 3.87
Anteil der Grünfläche: 15.47%
Punkte für Grünflächenanteil: 2
Anzahl der Kreuzungen: 1140
Connectivity_Score: 7
Gesamtlänge der Fahrradinfrastruktur: 57752.024999999994 Meter
Gesamtlänge aller primären und sekundären Straßen: 64564.514 Meter
Verhältnis: 89.4485552853383%
Bewertungspunkt: 10
Länge der Fahrradroute ohne Gewichtung: 424431.385
Länge der Fahrradroute: 428437.299
Länge Insgesamt: 490183.358
Prozentualer Share: 87.40347708826133
Bewertungspunkt: 10
Endbewertung: 8.179443647501852
latitude: 53.57257526671511 longitue: 10.135841171065165
Gewichteter Surfaces_Score: 8.517997186302505
Gewichteter Slope_Score: 8.101213432765483


C:\Users\kevdr\AppData\Local\Temp\ipykernel_20616\1858481351.py:196: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


...

Gesamtlänge der Fahrradinfrastruktur: 1441.612 Meter
Gesamtlänge aller primären und sekundären Straßen: 34456.481 Meter
Verhältnis: 4.183863117072228%
Bewertungspunkt: 2
Länge der Fahrradroute ohne Gewichtung: 237588.482
Länge der Fahrradroute: 237588.482
Länge Insgesamt: 392596.899
Prozentualer Share: 60.51715706496195
Bewertungspunkt: 6
Endbewertung: 5.081011574804002
latitude: 54.78414815256961 longitue: 9.362944894753715
Gewichteter Surfaces_Score: 7.8669428823664225
Gewichteter Slope_Score: 7.474441408817351


C:\Users\kevdr\AppData\Local\Temp\ipykernel_20616\1858481351.py:196: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in km²: 12.66
Anteil der Grünfläche: 50.54%
Punkte für Grünflächenanteil: 6
Anzahl der Kreuzungen: 923
Connectivity_Score: 6
Gesamtlänge der Fahrradinfrastruktur: 16451.942 Meter
Gesamtlänge aller primären und sekundären Straßen: 33075.397 Meter
Verhältnis: 49.740724200528874%
Bewertungspunkt: 6
Länge der Fahrradroute ohne Gewichtung: 363492.846
Länge der Fahrradroute: 364292.73199999996
Länge Insgesamt: 568017.666
Prozentualer Share: 64.13404966175823
Bewertungspunkt: 6
Endbewertung: 6.449124371942064
latitude: 54.78452039789875 longitue: 9.440631384859842
Gewichteter Surfaces_Score: 7.634676114020241
Gewichteter Slope_Score: 7.690957221031163
Grünfläche in km²: 10.04
Anteil der Grünfläche: 40.10%
Punkte für Grünflächenanteil: 4


C:\Users\kevdr\AppData\Local\Temp\ipykernel_20616\1858481351.py:196: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Anzahl der Kreuzungen: 1387
Connectivity_Score: 8
Gesamtlänge der Fahrradinfrastruktur: 5231.214 Meter
Gesamtlänge aller primären und sekundären Straßen: 30913.819 Meter
Verhältnis: 16.921927374938697%
Bewertungspunkt: 2
Länge der Fahrradroute ohne Gewichtung: 349753.634
Länge der Fahrradroute: 349753.634
Länge Insgesamt: 539265.437
Prozentualer Share: 64.8574171461317
Bewertungspunkt: 6
Endbewertung: 5.409241905766994


In [12]:
print(G_original)

MultiDiGraph with 1505 nodes and 2596 edges


Nach Endbwertung sortieren

In [2]:
import pandas as pd

# Pfad zur CSV-Datei
csv_datei_pfad = 'NeueGewichtung_Min20Data_Deutschland.csv'

# CSV-Datei laden
df = pd.read_csv(csv_datei_pfad)

# Daten nach 'Endbewertung' sortieren, absteigend, sodass die höchste Zahl zuerst kommt
df_sortiert = df.sort_values(by='Endbewertung', ascending=False)

# Die sortierte Tabelle ausgeben
print(df_sortiert)

# Optional: Die sortierte Tabelle in eine neue CSV-Datei speichern
df_sortiert.to_csv('NeueGewichtung_Min20Data_Deutschland.csv', index=False)


         GITTER_5km  Total_Count  Access_Count  Used_Count  TOT_P_2021  \
217   5kmN2775E4440         1317        1053.0       199.0    134765.0   
216   5kmN2775E4435         1739        1432.0       345.0    173644.0   
238   5kmN2780E4435         3078        2494.0       789.0    291449.0   
215   5kmN2775E4430         1625        1370.0       324.0    138900.0   
2486  5kmN3330E4240          784         694.0       261.0    101315.0   
...             ...          ...           ...         ...         ...   
2718  5kmN3400E4370           33          32.0         7.0      1290.0   
2732  5kmN3410E4225           34          32.0         2.0      1058.0   
2747  5kmN3410E4390           27          16.0         2.0       589.0   
2762  5kmN3420E4230           48          39.0        11.0      4059.0   
2775  5kmN3425E4330           49          37.0         2.0      5274.0   

            lat        lon  min_elevation  max_elevation  elevation_diff  ...  \
217   48.076997  11.596336    

NaN Hauptstraße aussortieren

In [3]:
import pandas as pd

# Pfad zur CSV-Datei
csv_datei_pfad = 'NeueGewichtung_Min20Data_Deutschland.csv'

# CSV-Datei laden
df = pd.read_csv(csv_datei_pfad)

# Zeilen löschen, die in der Spalte 'Hauptstraße' keinen Wert haben
df_rein = df.dropna(subset=['Hauptstraße_Bewertung'])

# Das Ergebnis ausgeben, um die Änderung zu überprüfen
print(df_rein)

# Optional: Speichere den bereinigten DataFrame in einer neuen CSV-Datei
df_rein.to_csv('NeueGewichtung_Min20Data_Deutschland.csv', index=False)


         GITTER_5km  Total_Count  Access_Count  Used_Count  TOT_P_2021  \
0     5kmN2775E4440         1317        1053.0       199.0    134765.0   
1     5kmN2775E4435         1739        1432.0       345.0    173644.0   
2     5kmN2780E4435         3078        2494.0       789.0    291449.0   
3     5kmN2775E4430         1625        1370.0       324.0    138900.0   
4     5kmN3330E4240          784         694.0       261.0    101315.0   
...             ...          ...           ...         ...         ...   
2765  5kmN3090E4495           32          27.0         2.0      1002.0   
2766  5kmN2970E4185           35          25.0         3.0     12540.0   
2767  5kmN3125E4385           40          31.0         3.0      2514.0   
2768  5kmN3020E4130           22          11.0         0.0      2106.0   
2769  5kmN3100E4430           23          16.0         0.0      1522.0   

            lat        lon  min_elevation  max_elevation  elevation_diff  ...  \
0     48.076997  11.596336    

In [1]:
import pandas as pd

# Pfad zur CSV-Datei
csv_datei_pfad = 'NeueGewichtung_Min20Data_Deutschland.csv'

# CSV-Datei laden
df = pd.read_csv(csv_datei_pfad)

# Stelle sicher, dass 'Total_Count' und 'TOT_P_2021' als numerische Spalten behandelt werden
df['Total_Count'] = pd.to_numeric(df['Total_Count'], errors='coerce')
df['TOT_P_2021'] = pd.to_numeric(df['TOT_P_2021'], errors='coerce')

# Filtere die Zeilen, für die [Total_Count]/[TOT_P_2021] > 0.001 ist
df_gefiltert = df[df['Total_Count'] / df['TOT_P_2021'] > 0.001]

# Berechne nun die Fahrradnutzungsrate für die gefilterten Daten
# Angenommen, die Fahrradnutzungsrate wird wie zuvor berechnet
# Zum Beispiel: (df_gefiltert['Taeglich'] + df_gefiltert['1-3TageninWoche']) / df_gefiltert['Total_Count']
# Ersetze 'Taeglich' und '1-3TageninWoche' durch die tatsächlichen Spaltennamen für tägliche Nutzung und Nutzung 1-3 Tage in der Woche, falls anders benannt

# Nur als Beispiel, da die tatsächlichen Spalten für die Berechnung nicht klar spezifiziert wurden
#df_gefiltert['Fahrradnutzungsrate[1-3Woche/Total_Count]'] = (df_gefiltert['Taeglich'] + df_gefiltert['1-3TageninWoche']) / df_gefiltert['Total_Count']
#df_gefiltert['Fahrradnutzungsrate[Used_Count/Total_Count]'] = df_gefiltert['Used_Count']/df_gefiltert['Total_Count']
# Ergebnis ausgeben (optional: nur die ersten paar Zeilen zur Überprüfung)
print(df_gefiltert.head())

# Optional: Speichere die gefilterten und berechneten Daten in einer neuen CSV-Datei
df_gefiltert.to_csv('Neu_Filterung_Gewichtung_Min20Data_Deutschland.csv', index=False)


      GITTER_5km  Total_Count  Access_Count  Used_Count  TOT_P_2021  \
0  5kmN2775E4440         1317        1053.0       199.0    134765.0   
1  5kmN2775E4435         1739        1432.0       345.0    173644.0   
2  5kmN2780E4435         3078        2494.0       789.0    291449.0   
3  5kmN2775E4430         1625        1370.0       324.0    138900.0   
4  5kmN3330E4240          784         694.0       261.0    101315.0   

         lat        lon  min_elevation  max_elevation  elevation_diff  ...  \
0  48.076997  11.596336            524            558              34  ...   
1  48.077949  11.529280            505            571              66  ...   
2  48.122928  11.530634            502            542              40  ...   
3  48.078860  11.462222            526            586              60  ...   
4  53.072355   8.791507              1             25              24  ...   

   keineAngabe  keineBefragung  average_slope  Hauptstraße_Bewertung  \
0            1             317  

Korrelation

In [40]:
import pandas as pd

# Pfad zur CSV-Datei
csv_datei_pfad = 'Min20Data_Deutschland.csv'

# CSV-Datei laden
df = pd.read_csv(csv_datei_pfad)

# Stelle sicher, dass die Spalten 'Taeglich', '1-3TageninWoche', und 'Total_Count' numerische Daten enthalten
# Konvertiere die Spalten in numerische Daten, falls notwendig
df['Taeglich'] = pd.to_numeric(df['Taeglich'], errors='coerce')
df['1-3TageninWoche'] = pd.to_numeric(df['1-3TageninWoche'], errors='coerce')
df['Total_Count'] = pd.to_numeric(df['Total_Count'], errors='coerce')

# Berechne die neue Spalte für die Fahrradnutzungsrate
#df['Fahrradnutzungsrate'] = (df['Taeglich'] + df['1-3TageninWoche']) / (df['Total_Count'])
#Used Count
df['Fahrradnutzungsrate'] = (df['Used_Count']) / df['Total_Count']

# Berechne die Pearson-Korrelation zwischen 'Fahrradnutzungsrate' und 'Endbewertung'
korrelation = df[['Fahrradnutzungsrate', 'Endbewertung']].corr()

# Die Korrelation ausgeben
print(korrelation)


                     Fahrradnutzungsrate  Endbewertung
Fahrradnutzungsrate             1.000000      0.231005
Endbewertung                    0.231005      1.000000


Regressionsanalyse(Mit Filterung Total Count)

In [11]:
import pandas as pd
import statsmodels.api as sm

# Pfad zur CSV-Datei
#csv_datei_pfad = 'Min20Data_Deutschland.csv'  #Ohne 0,01% Filterung
csv_datei_pfad = 'NeueGewichtung_Min20Data_Deutschland.csv' #Mit 0,01% Filterung
# CSV-Datei laden
df = pd.read_csv(csv_datei_pfad)
#df = df[df['Total_Count'] > 40]
#df = df[df['TOT_P_2021'] > 10000]
# Berechne die neue Spalte für die Fahrradnutzungsrate
df['Fahrradnutzungsrate'] = (df['Used_Count']) / (df['Total_Count'])
#df['Fahrradnutzungsrate'] = (df['Taeglich'] + df['1-3TageninWoche']) / (df['Total_Count'])

# Unabhängige Variable (X) und abhängige Variable (Y)
X = df['Endbewertung']  # Endbewertung als Prädiktor
Y = df['Fahrradnutzungsrate']  # Fahrradnutzungsrate als Zielvariable

# Eine Konstante zu X hinzufügen, um den Achsenabschnitt (Intercept) zu berücksichtigen
X = sm.add_constant(X)

# Das Modell erstellen
modell = sm.OLS(Y, X)

# Die Regression durchführen
ergebnisse = modell.fit()

# Die Ergebnisse ausgeben
print(ergebnisse.summary())


                             OLS Regression Results                            
Dep. Variable:     Fahrradnutzungsrate   R-squared:                       0.151
Model:                             OLS   Adj. R-squared:                  0.151
Method:                  Least Squares   F-statistic:                     493.8
Date:                 Fri, 08 Mar 2024   Prob (F-statistic):          8.22e-101
Time:                         19:14:02   Log-Likelihood:                 3215.9
No. Observations:                 2770   AIC:                            -6428.
Df Residuals:                     2768   BIC:                            -6416.
Df Model:                            1                                         
Covariance Type:             nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0306      0.007    

In [25]:
csv_datei_pfad = 'NeueGewichtung_Min20Data_Deutschland.csv' #Mit 0,01% Filterung
# CSV-Datei laden
df = pd.read_csv(csv_datei_pfad)

df['TOT_P_2021'].__len__()

2770